# FLUX.1 [schnell] — DIMER E2E text-to-image QLoRA fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/flux-schnell-generation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/flux-schnell-generation-pipeline/blob/main/tutorials/flux_schnell_generation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-black--forest--labs%2FFLUX.1--schnell-ffcc4d?style=flat)](https://huggingface.co/black-forest-labs/FLUX.1-schnell) [![Upstream](https://img.shields.io/badge/Upstream-black--forest--labs%2Fflux-181717?style=flat&logo=github&logoColor=white)](https://github.com/black-forest-labs/flux) [![Licence](https://img.shields.io/badge/weights-Apache--2.0-blue.svg)](https://huggingface.co/black-forest-labs/FLUX.1-schnell/blob/main/LICENSE.md)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** few-step text-to-image generation with a 12 B-parameter rectified-flow transformer loaded 4-bit, held-out flow-matching-loss and CLIP-scored evaluation, and bounded QLoRA fine-tuning to a set of captioned photographs

**This notebook is standalone.** It carries the repository's package (3 modules under `src/flux_schnell_generation_pipeline/`, at revision `77567b768674`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub (the upstream identity is `black-forest-labs/FLUX.1-schnell`; the bytes are served by the ungated mirror `unsloth/FLUX.1-schnell` at `9df3faa7…`, see Section 3) at the immutable upstream revision `741f7c3ce8b383c54771c7003378a50191e9efe9` (~34335 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh **GPU** runtime (a 16 GB T4 is enough; see the Prerequisites) installs the pinned dependencies (torch, diffusers, transformers, peft, bitsandbytes, accelerate, sentencepiece, safetensors, huggingface-hub, numpy, pillow), stages and digest-verifies two pinned snapshots — the 33.7 GB FLUX.1 [schnell] diffusers-layout snapshot (transformer, CLIP-L and T5-XXL encoders, tokenizers, VAE, scheduler), fetched from an ungated mirror and checked byte for byte against the manifest the repository committed, and a 0.6 GB CLIP scorer — fetches 60 CC0 iNaturalist bird photographs as digest-verified JPEGs (6 MB, no credential), validates them and splits them 36 / 12 / 12 by seed, encodes every prompt with the two text encoders and releases them, loads the 12 B transformer 4-bit (NF4) with an untrained LoRA adapter attached, scores the frozen model — the held-out flow-matching loss on the validation and test photographs, and six four-step generations scored by CLIP against their prompts, the held-out photographs and the real-photo ceiling — runs a bounded QLoRA fine-tuning (3 epochs over 36 images), scores the adapted model on identical inputs, renders a new prompt, exports the adapter as safetensors with a manifest, releases the adapted transformer and reloads the artifact into a fresh 4-bit pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a T4 the whole path takes about an hour of model time after the 34 GB of downloads; the timings recorded on the release run are in `docs/release-verification.md`.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own captioned photographs as a zip holding `captions.csv` (columns `id`, `file`, `caption`) beside the image files (JPEG or PNG, shorter side 256..4096 px; at least four images, and at least one caption with three or more images so a held-out record exists). Your records are split by caption into training, validation and test sets and flow through the same contract — validation, prompt encoding, frozen baseline, adaptation, held-out evaluation, generation, artifact export and reload parity. Because the text encoders and the transformer do not share the GPU, re-running from Section 4 releases the transformer before your prompts are encoded (Section 5 does this). The expected schema, the ceilings and the privacy guidance are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

FLUX.1 [schnell] (Black Forest Labs, 2024) is a 12 B-parameter rectified-flow transformer distilled to render an image in one to four sampling steps with no classifier-free guidance. A CLIP-L encoder gives one pooled vector, a T5-XXL encoder gives 256 token embeddings, 19 double-stream and 38 single-stream transformer blocks predict the *velocity* of a 16-channel latent — the FLUX VAE's 8× compressed image, packed 2×2 into 64-channel tokens — and the VAE decodes the latent to pixels. Everything comes from one 33.7 GB snapshot in the diffusers layout, plus, for evaluation only, a CLIP ViT-B/32 scorer.

Three things about this row are handled in the open. **The transformer does not fit a 16 GB GPU in any 16-bit format** (23.8 GB as shipped), so it is loaded 4-bit — bitsandbytes NF4 with double quantisation, float16 compute — at about 6.6 GB, and the LoRA is trained over that quantised base (QLoRA); the frozen numbers are therefore the 4-bit model's numbers, not the bfloat16 checkpoint's. **The text encoders (9.7 GB in float16) do not fit beside it**, so Section 5 encodes every prompt the notebook will ever use once, keeps the embeddings, and releases both encoders before the transformer loads; the reloaded pipeline in Section 9 adopts the same embeddings rather than loading them again. **Generation has no ground truth**, so the notebook reads three kinds of number and says what each is: the held-out *flow-matching loss* (the training objective, measured on photographs the model never trained on, with identical noise for the frozen and the adapted model), CLIP scores of generated images (prompt alignment, which species CLIP thinks it sees, and closeness to the held-out real photographs), and the same CLIP scores on the real photographs themselves — the ceiling. None of these is a human judgement of image quality.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and scorer modules; stage and digest-verify a pinned snapshot served by a mirror against a committed manifest; fetch, validate and split a small real captioned-photograph dataset; encode prompts with two text encoders and release them; load a 12 B transformer 4-bit; read a held-out flow-matching loss and CLIP-scored few-step generations against a real-photo ceiling; run a bounded QLoRA fine-tuning of a rectified-flow transformer with explicit hyperparameters; compare the adapted and frozen models on identical held-out inputs; render a new prompt; and export a safetensors adapter that reloads against the pinned 4-bit base with verified parity.

**This notebook does not demonstrate:** FLUX.1 [dev] or [pro], guidance-distillation or classifier-free guidance (schnell has none), ControlNet, inpainting or image-to-image conditioning, resolutions other than 512², full or 16-bit fine-tuning, DreamBooth identifiers, safety filtering of prompts or images, human preference or FID/KID benchmarks, prompt engineering, and any claim that a CLIP score or a flow-matching loss measures image quality. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported **GPU** runtime (Google Colab T4 or better, or a Jupyter kernel with a CUDA GPU of at least 15 GB, bitsandbytes support and Python 3.12). The CLIP-L and T5-XXL encoders run in float16 (9.7 GB) while they encode prompts and are then released; the transformer is loaded 4-bit (NF4, about 6.6 GB) with float16 compute and the LoRA parameters in float32; the FLUX VAE stays in float32 (0.34 GB). CPU-only runtimes are not supported for this notebook: bitsandbytes 4-bit needs CUDA and the 12 B transformer would need 48 GB of RAM in float32. About 36 GB of disk is needed for the two snapshots.
- **Knowledge:** what a latent diffusion or flow-matching model does at inference (noise → latent → image), why a distilled few-step model uses no classifier-free guidance, what 4-bit weight quantisation changes, what a LoRA adapter changes and what it does not, and why a training loss is not a quality score.
- **Weights:** the transformer, both text encoders, the VAE and the CLIP scorer are all safetensors; nothing is unpickled and no Hub-hosted code is executed — the model classes come from `diffusers`, `transformers`, `peft` and `bitsandbytes` on PyPI. FLUX.1 [schnell] is released under the Apache-2.0 licence; the scorer is MIT. The upstream Hub repository is gated by a click-through, so the bytes are fetched from an ungated, unmodified mirror and verified against the manifest committed in this repository (Section 3).
- **Data contract:** a record is `{{id, image, caption}}` — an RGB image with shorter side 256..4096 px (resized so the shorter side is 512 px and centre-cropped to 512 × 512; the crop is reported) and a caption of 1..1000 characters (truncated to 256 T5 tokens and 77 CLIP tokens; T5 truncations are reported). Validation is structural: nothing checks that a caption describes its image or that the model can render it.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — photographs of identifiable people, licensed stock images or client material are exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches 60 pinned photographs (about 6 MB) from the public iNaturalist open-data bucket `inaturalist-open-data.s3.amazonaws.com` over HTTPS, digest-verified before decoding; every photo is CC0 and its observation page is recorded.
- **External access:** the Hugging Face Hub (the upstream identity is `black-forest-labs/FLUX.1-schnell`; the bytes are served by the ungated mirror `unsloth/FLUX.1-schnell` at `9df3faa7…`, see Section 3) only, to fetch the pinned `black-forest-labs/FLUX.1-schnell` snapshot (~34335 MB in total) at revision `741f7c3ce8b3…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `diffusers`, `transformers`, `peft`, `bitsandbytes` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'diffusers==0.40.0',
    'transformers==5.17.0',
    'peft==0.21.0',
    'bitsandbytes==0.50.2',
    'torchao==0.18.0',
    'accelerate==1.15.0',
    'tokenizers==0.23.2',
    'sentencepiece==0.2.2',
    'protobuf==7.36.2',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'flux-schnell-generation-pipeline',
    'repository_revision': '77567b768674bce3fb402388c1fb7053bbfc8fe0',
    'embedded_module': 'src/flux_schnell_generation_pipeline/pipeline.py',
    'embedded_modules': ['src/flux_schnell_generation_pipeline/pipeline.py', 'src/flux_schnell_generation_pipeline/metrics.py', 'src/flux_schnell_generation_pipeline/samples.py'],
    'module_sha256': 'f95ff5250d5df24155069e438282bae2a03e8b45733d50e4c49772593cbac149',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, diffusers, transformers, peft, bitsandbytes
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'diffusers': diffusers.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'bitsandbytes': bitsandbytes.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/flux_schnell_generation_pipeline/` @ `77567b768674`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/flux_schnell_generation_pipeline/pipeline.py`

In [ ]:
"""FLUX.1 [schnell] (`black-forest-labs/FLUX.1-schnell`) DIMER pipeline: a verified snapshot, few-step text-to-image
generation, held-out flow-matching-loss evaluation, and bounded QLoRA fine-tuning of the 4-bit diffusion transformer to
a user's captioned images with a portable adapter.

FLUX.1 [schnell] (Black Forest Labs, 2024) is a 12 B-parameter rectified-flow transformer distilled to generate in one to
four sampling steps without classifier-free guidance. A CLIP-L encoder gives one pooled vector, a T5-XXL encoder gives
256 token embeddings, 19 double-stream and 38 single-stream transformer blocks predict the flow velocity of a 16-channel
latent (the FLUX VAE's 8× downsampled image, packed 2×2 into 64-channel tokens), and the VAE decodes the latent to
pixels. Everything comes from **one pinned snapshot** in the diffusers layout (23 files, 33.7 GB, bfloat16 as shipped):

* the identity of record is the upstream repository `MODEL_ID` at `MODEL_REVISION` (Apache-2.0), whose Hub page is
  gated behind a click-through; the files are **staged from the ungated mirror** `STAGING_ID` at `STAGING_REVISION`,
  whose 23 diffusers-layout files match the upstream tree file for file and byte for byte in size (upstream LFS
  digests are hidden behind the gate; see `docs/WEIGHTS.md` for the state of the byte-identity proof);
* the transformer (23.8 GB) is too large for a 16 GB GPU in any 16-bit format, so it is **always loaded 4-bit**
  (bitsandbytes NF4 with double quantisation; about 6.6 GB) with `COMPUTE_DTYPE` compute — a GPU is required;
* the text encoders (9.7 GB in 16-bit) are loaded only to encode prompts and released before the transformer loads;
* the **scorer** used only by evaluation (`SCORER_ID`, an MIT-licensed CLIP ViT-B/32, 605 MB) is a second snapshot.

Every file is safetensors or plain JSON/text: nothing is unpickled and no Hub-hosted code is executed (the model
classes come from `diffusers` and `transformers` on PyPI).

The adaptation contract is LoRA (rank 8) on the image-stream query/key/value/output projections of all 57 blocks
(380 tensors, 9,338,880 parameters) over the frozen 4-bit base — QLoRA; the VAE and text encoders stay frozen.
Training minimises the rectified-flow velocity MSE on the user's images; the held-out metric is the same MSE at fixed
noise levels and fixed noise, so the frozen and adapted models are compared on identical inputs. Everything
model-related is imported lazily so that snapshot verification and input validation run (and can refuse) before
`torch`, `diffusers`, `transformers` or `bitsandbytes` are imported (fleet RTM-001).
"""

from __future__ import annotations

import contextlib
import gc
import hashlib
import json
import math
import time
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "black-forest-labs/FLUX.1-schnell"
MODEL_REVISION = "741f7c3ce8b383c54771c7003378a50191e9efe9"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "flux1-schnell"
# The upstream repository is gated (click-through); the files are fetched from this unmodified, ungated mirror.
STAGING_ID = "unsloth/FLUX.1-schnell"
STAGING_REVISION = "9df3faa7ae3b6ddf0b2b69bb78616372897cc65c"
ARTIFACT_FORMAT = "org.valcorza.flux-schnell-generation.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory-relative
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / MODEL_KEY
MANIFEST_NAME = "dimer-base-manifest.json"
# Evaluation-only scorer (never trained, never part of generation): a CLIP ViT-B/32 served as safetensors.
SCORER_ID = "laion/CLIP-ViT-B-32-laion2B-s34B-b79K"
SCORER_REVISION = "1a25a446712ba5ee05982a381eed697ef9b435cf"
SCORER_LICENSE = "mit"
SCORER_KEY = "clip-vit-b-32-laion2b"
SCORER_WEIGHTS_DIR = _WEIGHTS_ROOT / SCORER_KEY

# Architecture and contract facts (transformer/config.json and the safetensors headers of the pinned snapshot).
TRANSFORMER_PARAMETERS = 11_891_178_560
TRANSFORMER_TENSORS = 1_156
NUM_DOUBLE_BLOCKS = 19
NUM_SINGLE_BLOCKS = 38
HIDDEN_SIZE = 3072
T5_HIDDEN = 4096
POOLED_DIM = 768
RESOLUTION = 512  # generation and training resolution here (the model was trained at up to 2 MP); centre-cropped
LATENT_CHANNELS = 16
VAE_SCALE = 8
PACKED_CHANNELS = LATENT_CHANNELS * 4  # 2×2 latent patches -> 64-channel tokens
MAX_PROMPT_TOKENS = 256  # the T5 sequence length FLUX.1 [schnell] was distilled with
MAX_CAPTION_CHARS = 1_000
MIN_IMAGE_SIDE = 256
MAX_IMAGE_SIDE = 4_096
MIN_TRAIN_RECORDS = 4
MAX_RECORDS = 2_000
DEFAULT_STEPS = 4
MAX_STEPS = 8  # schnell is distilled for 1-4 steps; more is allowed for inspection only
GUIDANCE_SCALE = 0.0  # guidance-distilled: `guidance_embeds` is false and classifier-free guidance is not used
EVAL_SIGMAS: tuple[float, ...] = (0.1, 0.3, 0.5, 0.7, 0.9)  # fixed noise levels of the held-out flow-matching loss
QUANTIZATION = "nf4"  # bitsandbytes 4-bit NormalFloat, double quantisation; the only way a 12 B transformer fits 16 GB
COMPUTE_DTYPES: tuple[str, ...] = ("float16", "bfloat16")
COMPUTE_DTYPE = "float16"  # default compute/activation dtype; the T4 has no native bfloat16 (see docs/WEIGHTS.md)
LORA_RANK = 8
LORA_ALPHA = 8
LORA_TARGETS: tuple[str, ...] = ("to_q", "to_k", "to_v", "to_out.0")
LORA_MODULES = 3 * (NUM_DOUBLE_BLOCKS + NUM_SINGLE_BLOCKS) + NUM_DOUBLE_BLOCKS  # single blocks have no to_out.0
LORA_TENSORS = 2 * LORA_MODULES  # 380: (A, B) per module
LORA_PARAMETERS = LORA_MODULES * 2 * LORA_RANK * HIDDEN_SIZE  # 9,338,880
SNAPSHOT_FILE_SUFFIXES = (".safetensors", ".json", ".model", ".txt", ".md")  # the scorer snapshot carries its README


# --------------------------------------------------------------------------------------------------
# manifests and staging (the model snapshot, staged from the mirror, and the scorer snapshot)
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if not entry["path"].endswith(SNAPSHOT_FILE_SUFFIXES):
            raise ValueError(f"{entry['path']}: unexpected file type in a code-free snapshot")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the model snapshot against its DIMER manifest (size + SHA-256 of every listed file). The manifest names
    the upstream identity (`MODEL_ID`@`MODEL_REVISION`) and the mirror the bytes were staged from."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    staging = manifest.get("staging", {})
    if (staging.get("repo"), staging.get("revision")) != (STAGING_ID, STAGING_REVISION):
        raise ValueError(f"manifest staging {staging!r} != {STAGING_ID}@{STAGING_REVISION}")
    return manifest


def verify_scorer_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the CLIP scorer snapshot against its own manifest."""
    root = Path(path) if path is not None else SCORER_WEIGHTS_DIR
    return _verify_manifest(root, SCORER_ID, SCORER_REVISION)


def _hub_download(relative_path: str, root: Path, repo_id: str, revision: str) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(repo_id, relative_path, revision=revision, local_dir=str(root))


def _stage_missing(
    root: Path,
    model_id: str,
    revision: str,
    allow_download: bool,
    downloader: Callable[[str, Path], None] | None,
    *,
    source: tuple[str, str] | None = None,
) -> list[str]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {model_id}@{revision}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {revision}"
        )
    repo_id, repo_revision = source or (model_id, revision)
    fetch = downloader or (lambda rel, dst: _hub_download(rel, dst, repo_id, repo_revision))
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally from the ungated mirror `STAGING_ID`@`STAGING_REVISION` (a
    fresh clone commits the manifest and the small JSON/tokenizer files and git-ignores the 33.6 GB of safetensors).
    `verify_snapshot` then checks every byte against the manifest, so which host served a file cannot change what
    is loaded."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _stage_missing(
        root, MODEL_ID, MODEL_REVISION, allow_download, downloader, source=(STAGING_ID, STAGING_REVISION)
    )


def stage_missing_scorer_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Same for the CLIP scorer snapshot at SCORER_REVISION."""
    root = Path(path) if path is not None else SCORER_WEIGHTS_DIR
    return _stage_missing(root, SCORER_ID, SCORER_REVISION, allow_download, downloader)


# --------------------------------------------------------------------------------------------------
# captioned-image records and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "record": "{id, image, caption}: a PIL image (or a path to one) and the caption used to generate it",
    "image_side": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "resolution": RESOLUTION,
    "preprocessing": (
        f"each image is resized so its shorter side is {RESOLUTION} px and centre-cropped to {RESOLUTION}×{RESOLUTION}; "
        "the crop is reported per record (VAL7). Nothing else is changed"
    ),
    "caption_chars": [1, MAX_CAPTION_CHARS],
    "prompt_tokens": MAX_PROMPT_TOKENS,
    "records": [MIN_TRAIN_RECORDS, MAX_RECORDS],
    "generation": {"steps": [1, MAX_STEPS], "guidance_scale": GUIDANCE_SCALE, "size": RESOLUTION},
    "validation": (
        "record shape, image decodability and side limits, caption length and duplicate ids only. Nothing checks "
        "that a caption describes its image, that the images are photographs, or that the prompt is one the model "
        "can render -- any RGB image with any string is accepted"
    ),
}


def _check_record(record: Any, index: int) -> dict[str, Any]:
    from PIL import Image

    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/image/caption")
    for key in ("id", "image", "caption"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, image, caption = record["id"], record["image"], record["caption"]
    if not isinstance(rid, str) or not rid or len(rid) > 64:
        raise ValueError(f"{label}: id must be a non-empty string of at most 64 characters")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"{label}: image sides must be within {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px, got {image.size}")
    if not isinstance(caption, str) or not caption.strip() or len(caption) > MAX_CAPTION_CHARS:
        raise ValueError(f"{label}: caption must be a non-empty string of at most {MAX_CAPTION_CHARS} characters")
    item = {"id": rid, "image": image.convert("RGB"), "caption": caption.strip()}
    for key in ("label", "common_name", "scientific_name", "observer", "inat_photo_id", "inat_observation_url", "source_id"):
        if key in record:
            item[key] = record[key]
    return item


def image_digest(image: Any) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same photo matches."""
    rgb = image.convert("RGB")
    return hashlib.sha256(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes()).hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], image_digest(r["image"]), r["caption"]] for r in records]
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_TRAIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a captioned-image dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, caption} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    crops = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        width, height = item["image"].size
        if width != height:
            crops += 1
        checked.append(item)
    sides = [min(r["image"].size) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "n_captions": len({r["caption"] for r in checked}),
        "shorter_side": {"min": min(sides), "max": max(sides)},
        "centre_cropped": crops,
        "resolution": RESOLUTION,
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def validate_inputs(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record; returns its id, size, the crop it will get and the caption length."""
    item = _check_record(record, 0)
    width, height = item["image"].size
    short = min(width, height)
    scale = RESOLUTION / short
    return {
        "id": item["id"],
        "size": (width, height),
        "resized_to": (round(width * scale), round(height * scale)),
        "centre_crop": (RESOLUTION, RESOLUTION),
        "caption_chars": len(item["caption"]),
    }


def validate_prompts(prompts: Sequence[str]) -> list[str]:
    """Generation prompts: non-empty strings within the caption limit; duplicates are allowed."""
    if isinstance(prompts, str) or not isinstance(prompts, Sequence) or not prompts:
        raise ValueError("prompts must be a non-empty list of strings")
    out = []
    for index, prompt in enumerate(prompts):
        if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > MAX_CAPTION_CHARS:
            raise ValueError(f"prompts[{index}] must be a non-empty string of at most {MAX_CAPTION_CHARS} characters")
        out.append(prompt.strip())
    return out


def preprocess_image(image: Any) -> Any:
    """Resize the shorter side to RESOLUTION and centre-crop; returns a PIL RGB image of RESOLUTION²."""
    from PIL import Image

    rgb = image.convert("RGB")
    width, height = rgb.size
    scale = RESOLUTION / min(width, height)
    new = (max(RESOLUTION, round(width * scale)), max(RESOLUTION, round(height * scale)))
    resized = rgb.resize(new, Image.Resampling.BICUBIC)
    left = (new[0] - RESOLUTION) // 2
    top = (new[1] - RESOLUTION) // 2
    return resized.crop((left, top, left + RESOLUTION, top + RESOLUTION))


# --------------------------------------------------------------------------------------------------
# model construction
# --------------------------------------------------------------------------------------------------


def _preload_nvidia_libs() -> None:
    """Preload the pip-bundled NVIDIA runtime libraries so bitsandbytes resolves libnvJitLink etc. on hosted images
    whose system CUDA is older than the one torch was built against (fleet trap F8)."""
    import ctypes
    import os
    import sys

    for site_pkg in sys.path:
        nvidia_dir = Path(site_pkg) / "nvidia"
        if nvidia_dir.is_dir():
            libs = [str(p) for p in nvidia_dir.glob("*/lib")]
            if libs:
                existing = os.environ.get("LD_LIBRARY_PATH", "")
                prefix = ":".join(libs)
                os.environ["LD_LIBRARY_PATH"] = f"{prefix}:{existing}" if existing else prefix
            for so in nvidia_dir.rglob("lib*.so*"):
                with contextlib.suppress(Exception):
                    ctypes.CDLL(str(so), mode=getattr(ctypes, "RTLD_GLOBAL", 0))


def _torch_dtype(name: str) -> Any:
    import torch

    if name not in COMPUTE_DTYPES:
        raise ValueError(f"compute_dtype must be one of {COMPUTE_DTYPES}")
    return getattr(torch, name)


def _lora_config() -> Any:
    from peft import LoraConfig

    return LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, init_lora_weights="gaussian", target_modules=list(LORA_TARGETS))


def lora_parameter_names(transformer: Any) -> list[str]:
    """The exact tensor set the adaptation contract may change on a transformer built with the adapter."""
    return sorted(name for name, _param in transformer.named_parameters() if ".lora_A." in name or ".lora_B." in name)


def count_parameters(model: Any) -> int:
    """Parameters as the checkpoint counts them: a 4-bit packed tensor counts the elements of its original shape."""
    total = 0
    for param in model.parameters():
        quant_state = getattr(param, "quant_state", None)
        total += math.prod(quant_state.shape) if quant_state is not None else param.numel()
    return total


def build_transformer(weights_dir: Path, *, dtype: Any, use_lora: bool, device: str = "cuda") -> Any:
    """Load the pinned transformer 4-bit (NF4, double quantisation, `dtype` compute) from the verified snapshot onto
    `device`, check the parameter count against the checkpoint, freeze it, and optionally attach the (untrained) LoRA."""
    _preload_nvidia_libs()
    import torch
    from diffusers import BitsAndBytesConfig, FluxTransformer2DModel

    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type=QUANTIZATION, bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = FluxTransformer2DModel.from_pretrained(
            str(weights_dir), subfolder="transformer", quantization_config=quant, torch_dtype=dtype
        ).to(device)
    n_params = count_parameters(model)
    if n_params != TRANSFORMER_PARAMETERS:
        raise ValueError(f"transformer has {n_params} parameters; expected {TRANSFORMER_PARAMETERS}")
    for param in model.parameters():
        param.requires_grad_(False)
    if use_lora:
        from peft import inject_adapter_in_model

        inject_adapter_in_model(_lora_config(), model, adapter_name="default")
        names = lora_parameter_names(model)
        if len(names) != LORA_TENSORS:
            raise ValueError(f"adapter attached {len(names)} LoRA tensors, expected {LORA_TENSORS}")
        name_set = set(names)
        for name, param in model.named_parameters():
            if name in name_set:
                param.data = param.data.to(torch.float32)  # trained in float32 under autocast
                param.requires_grad_(False)
    model.eval()
    return model


def _select_device(device: str | None) -> str:
    import torch

    if device is None:
        if not torch.cuda.is_available():
            raise ValueError(
                "FLUX.1 [schnell] is loaded 4-bit with bitsandbytes and needs a CUDA GPU; validation, staging and "
                "verification work without one"
            )
        return "cuda"
    if not device.startswith("cuda"):
        raise ValueError("only CUDA devices are supported: the 12 B transformer is loaded 4-bit with bitsandbytes")
    if not torch.cuda.is_available():
        raise ValueError("device='cuda' requested but CUDA is not available")
    return device


def pack_latents(latents: Any) -> Any:
    """(B, 16, H, W) latents -> (B, H/2 · W/2, 64) tokens: each 2×2 latent patch becomes one token."""
    batch, channels, height, width = latents.shape
    x = latents.view(batch, channels, height // 2, 2, width // 2, 2)
    return x.permute(0, 2, 4, 1, 3, 5).reshape(batch, (height // 2) * (width // 2), channels * 4)


def unpack_latents(tokens: Any, height: int, width: int) -> Any:
    """Inverse of `pack_latents` for a (height, width) latent grid."""
    batch = tokens.shape[0]
    x = tokens.view(batch, height // 2, width // 2, LATENT_CHANNELS, 2, 2)
    return x.permute(0, 3, 1, 4, 2, 5).reshape(batch, LATENT_CHANNELS, height, width)


def latent_image_ids(height: int, width: int, device: Any, dtype: Any) -> Any:
    """Rotary position ids of the packed latent tokens: (H/2 · W/2, 3) rows of (0, row, column)."""
    import torch

    ids = torch.zeros(height // 2, width // 2, 3)
    ids[..., 1] += torch.arange(height // 2)[:, None]
    ids[..., 2] += torch.arange(width // 2)[None, :]
    return ids.reshape(-1, 3).to(device=device, dtype=dtype)


@dataclass
class FluxSchnellPipeline:
    """Few-step text-to-image generation and bounded QLoRA fine-tuning on top of the verified FLUX.1 [schnell]
    snapshot. The transformer (4-bit) and the text encoders (16-bit) do not fit a 16 GB GPU together, so prompts
    are encoded first (`encode_prompts`, which loads CLIP-L and T5-XXL) and the transformer is loaded on first use
    by `generate` / `evaluate` / `adapt`, which release the text encoders if they are still resident."""

    vae: Any
    scheduler_config: dict[str, Any]
    tokenizer: Any
    tokenizer_2: Any
    device: str
    dtype: Any
    compute_dtype: str
    weights_dir: Path
    source: str
    use_lora: bool
    transformer: Any = None
    adapter: dict[str, Any] | None = None
    _text_encoder: Any = field(default=None, repr=False)
    _text_encoder_2: Any = field(default=None, repr=False)
    _prompt_cache: dict[str, Any] = field(default_factory=dict, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        use_lora: bool = False,
        compute_dtype: str = COMPUTE_DTYPE,
        load_transformer: bool = False,
    ) -> FluxSchnellPipeline:
        """Stage (when allowed) and verify the snapshot, then load the VAE, both tokenizers and the scheduler
        config. The transformer loads lazily (or now, with `load_transformer=True`) and the text encoders only
        inside `encode_prompts`, because 6.6 GB (4-bit transformer) + 9.7 GB (16-bit encoders) exceed 16 GB."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        import torch
        from diffusers import AutoencoderKL, FlowMatchEulerDiscreteScheduler
        from transformers import AutoTokenizer

        chosen = _select_device(device)
        dtype = _torch_dtype(compute_dtype)
        # The FLUX VAE declares `force_upcast`; it is small (168 MB) and kept in float32 throughout.
        vae = AutoencoderKL.from_pretrained(str(root), subfolder="vae", torch_dtype=torch.float32).to(chosen).eval()
        for param in vae.parameters():
            param.requires_grad_(False)
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(str(root), subfolder="scheduler")
        pipeline = cls(
            vae=vae,
            scheduler_config=dict(scheduler.config),
            tokenizer=AutoTokenizer.from_pretrained(str(root / "tokenizer")),
            tokenizer_2=AutoTokenizer.from_pretrained(str(root / "tokenizer_2")),
            device=chosen,
            dtype=dtype,
            compute_dtype=compute_dtype,
            weights_dir=root,
            source=f"local-snapshot (manifest verified; safetensors only; staged from {STAGING_ID})",
            use_lora=use_lora,
        )
        if load_transformer:
            pipeline.load_transformer()
        return pipeline

    # ---- residency --------------------------------------------------------------------------------------

    def load_transformer(self) -> dict[str, Any]:
        """Load the 4-bit transformer onto the device, releasing the text encoders first if they are resident."""
        released = self.release_text_encoder()
        if self.transformer is not None:
            return {"loaded": False, "released_text_encoders": released}
        started = time.perf_counter()
        self.transformer = build_transformer(self.weights_dir, dtype=self.dtype, use_lora=self.use_lora, device=self.device)
        return {"loaded": True, "released_text_encoders": released, "load_seconds": round(time.perf_counter() - started, 1)}

    def release_transformer(self) -> bool:
        """Drop the transformer (and any adapter state it carries) so the text encoders can be loaded again."""
        import torch

        had = self.transformer is not None
        self.transformer = None
        self.adapter = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return had

    def _require_transformer(self) -> Any:
        if self.transformer is None:
            self.load_transformer()
        return self.transformer

    # ---- prompts ---------------------------------------------------------------------------------------

    def encode_prompts(self, prompts: Sequence[str]) -> dict[str, Any]:
        """Encode every distinct prompt with CLIP-L (pooled vector) and T5-XXL (256 token embeddings) into the
        pipeline's prompt cache. The encoders are loaded on first use in the compute dtype and stay loaded until
        `release_text_encoder` (or until the transformer loads). Refused while the transformer is resident: the
        two do not fit one 16 GB GPU together — encode every prompt first, or `release_transformer()`."""
        import torch
        from transformers import CLIPTextModel, T5EncoderModel

        wanted = [p for p in dict.fromkeys(validate_prompts(list(prompts))) if p not in self._prompt_cache]
        if not wanted:
            return {"encoded": 0, "cached": len(self._prompt_cache)}
        if self.transformer is not None:
            raise ValueError(
                "the transformer is resident; encode all prompts before generate/evaluate/adapt or call "
                "release_transformer() first (text encoders and the 4-bit transformer do not fit 16 GB together)"
            )
        if self._text_encoder is None:
            started = time.perf_counter()
            self._text_encoder = (
                CLIPTextModel.from_pretrained(str(self.weights_dir), subfolder="text_encoder", torch_dtype=self.dtype)
                .to(self.device)
                .eval()
            )
            self._text_encoder_2 = (
                T5EncoderModel.from_pretrained(str(self.weights_dir), subfolder="text_encoder_2", torch_dtype=self.dtype)
                .to(self.device)
                .eval()
            )
            load_seconds = round(time.perf_counter() - started, 1)
        else:
            load_seconds = 0.0
        started = time.perf_counter()
        with torch.inference_mode():
            for prompt in wanted:
                clip_tokens = self.tokenizer(
                    prompt, padding="max_length", max_length=self.tokenizer.model_max_length, truncation=True, return_tensors="pt"
                )
                pooled = self._text_encoder(clip_tokens.input_ids.to(self.device), output_hidden_states=False).pooler_output
                t5_tokens = self.tokenizer_2(
                    prompt,
                    padding="max_length",
                    max_length=MAX_PROMPT_TOKENS,
                    truncation=True,
                    return_length=False,
                    return_overflowing_tokens=False,
                    return_tensors="pt",
                )
                n_tokens = int(t5_tokens.attention_mask.sum())
                embeds = self._text_encoder_2(t5_tokens.input_ids.to(self.device), output_hidden_states=False)[0]
                self._prompt_cache[prompt] = {
                    "embeds": embeds[0].to("cpu", self.dtype),
                    "pooled": pooled[0].to("cpu", self.dtype),
                    "tokens": n_tokens,
                }
        return {
            "encoded": len(wanted),
            "cached": len(self._prompt_cache),
            "encoder_dtype": self.compute_dtype,
            "load_seconds": load_seconds,
            "encode_seconds": round(time.perf_counter() - started, 1),
            "truncated": [p[:40] for p in wanted if self._prompt_cache[p]["tokens"] >= MAX_PROMPT_TOKENS],
        }

    def export_prompt_cache(self) -> dict[str, Any]:
        """The encoded prompts (CPU tensors keyed by prompt), so a second pipeline can reuse them without loading
        the 9.7 GB text encoders again (the notebook's fresh-reload step)."""
        return {
            k: {"embeds": v["embeds"].clone(), "pooled": v["pooled"].clone(), "tokens": v["tokens"]}
            for k, v in self._prompt_cache.items()
        }

    def import_prompt_cache(self, cache: Mapping[str, Mapping[str, Any]]) -> int:
        """Adopt prompt embeddings exported by `export_prompt_cache` from a pipeline of the same identity."""
        for prompt, entry in cache.items():
            if tuple(entry["embeds"].shape) != (MAX_PROMPT_TOKENS, T5_HIDDEN) or tuple(entry["pooled"].shape) != (POOLED_DIM,):
                raise ValueError(f"prompt cache entry for {prompt[:40]!r} has an unexpected shape")
            self._prompt_cache[prompt] = {
                "embeds": entry["embeds"].to(self.dtype),
                "pooled": entry["pooled"].to(self.dtype),
                "tokens": int(entry["tokens"]),
            }
        return len(self._prompt_cache)

    def release_text_encoder(self) -> bool:
        """Drop both text encoders (the cached embeddings remain). Returns whether anything was released."""
        import torch

        had = self._text_encoder is not None or self._text_encoder_2 is not None
        self._text_encoder = None
        self._text_encoder_2 = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return had

    def _embeds(self, prompt: str) -> tuple[Any, Any]:
        if prompt not in self._prompt_cache:
            raise ValueError(f"prompt not encoded; call encode_prompts([...]) first: {prompt[:60]!r}")
        entry = self._prompt_cache[prompt]
        return entry["embeds"].to(self.device, self.dtype), entry["pooled"].to(self.device, self.dtype)

    def _batch_embeds(self, prompts: Sequence[str]) -> tuple[Any, Any]:
        import torch

        pairs = [self._embeds(p) for p in prompts]
        return torch.stack([e for e, _ in pairs]), torch.stack([p for _, p in pairs])

    # ---- generation ------------------------------------------------------------------------------------

    def _diffusers_pipeline(self) -> Any:
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers import FluxPipeline as _Upstream

        return _Upstream(
            scheduler=FlowMatchEulerDiscreteScheduler.from_config(self.scheduler_config),
            vae=self.vae,
            text_encoder=None,
            tokenizer=self.tokenizer,
            text_encoder_2=None,
            tokenizer_2=self.tokenizer_2,
            transformer=self._require_transformer(),
        )

    def generate(self, prompts: Sequence[str], *, seed: int = 0, steps: int = DEFAULT_STEPS) -> dict[str, Any]:
        """Generate one RESOLUTION² image per prompt in `steps` Euler steps without guidance; image i uses seed + i.
        Every prompt must already be encoded (`encode_prompts`)."""
        import numpy as np
        import torch

        prompts = validate_prompts(prompts)
        if not isinstance(steps, int) or not 1 <= steps <= MAX_STEPS:
            raise ValueError(f"steps must be an int in 1..{MAX_STEPS}")
        for prompt in prompts:
            self._embeds(prompt)
        pipe = self._diffusers_pipeline()
        pipe.set_progress_bar_config(disable=True)
        started = time.perf_counter()
        images = []
        for index, prompt in enumerate(prompts):
            embeds, pooled = self._embeds(prompt)
            generator = torch.Generator(device="cpu").manual_seed(seed + index)
            with torch.inference_mode():
                out = pipe(
                    prompt=None,
                    prompt_embeds=embeds[None],
                    pooled_prompt_embeds=pooled[None],
                    num_inference_steps=steps,
                    guidance_scale=GUIDANCE_SCALE,
                    height=RESOLUTION,
                    width=RESOLUTION,
                    max_sequence_length=MAX_PROMPT_TOKENS,
                    generator=generator,
                    output_type="latent",
                )
                image = self._decode(out.images)[0]
            array = np.asarray(image)
            images.append({"prompt": prompt, "seed": seed + index, "image": image, "pixel_mean": round(float(array.mean()), 3)})
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "steps": steps,
            "guidance_scale": GUIDANCE_SCALE,
            "size": (RESOLUTION, RESOLUTION),
            "scheduler": "FlowMatchEulerDiscreteScheduler (upstream config)",
            "precision": f"{QUANTIZATION} weights, {self.compute_dtype} compute",
            "images": images,
            "seconds": round(time.perf_counter() - started, 2),
        }

    # ---- latents and the flow-matching loss -------------------------------------------------------------

    def _decode(self, tokens: Any) -> list[Any]:
        """Packed latent tokens -> PIL images through the float32 VAE (undoing the shift/scale of `_latents`)."""
        import numpy as np
        import torch
        from PIL import Image

        grid = RESOLUTION // VAE_SCALE
        latents = unpack_latents(tokens.float(), grid, grid) / self.vae.config.scaling_factor + self.vae.config.shift_factor
        pixels = self.vae.decode(latents, return_dict=False)[0]
        arrays = ((pixels.float().clamp(-1, 1) + 1) * 127.5).round().permute(0, 2, 3, 1).to("cpu", torch.uint8).numpy()
        return [Image.fromarray(np.ascontiguousarray(a)) for a in arrays]

    def _latents(self, records: Sequence[Mapping[str, Any]], *, seed: int) -> Any:
        """VAE-encode preprocessed images to shifted, scaled latents (B, 16, 64, 64); the posterior sample is seeded."""
        import numpy as np
        import torch

        arrays = [np.asarray(preprocess_image(r["image"]), dtype=np.float32) / 127.5 - 1.0 for r in records]
        pixels = torch.from_numpy(np.stack(arrays)).permute(0, 3, 1, 2).to(self.device, torch.float32)
        generator = torch.Generator(device="cpu").manual_seed(seed)
        with torch.no_grad():
            posterior = self.vae.encode(pixels).latent_dist
            latents = (posterior.sample(generator=generator) - self.vae.config.shift_factor) * self.vae.config.scaling_factor
        return latents.to(self.dtype)

    def _autocast(self) -> Any:
        """Mixed precision for the compute dtype (a no-op for the float32 stubs of the offline tests)."""
        import torch

        return torch.autocast(device_type=self.device.split(":")[0], dtype=self.dtype, enabled=self.dtype != torch.float32)

    def _predict_velocity(self, noisy_tokens: Any, sigmas: Any, embeds: Any, pooled: Any) -> Any:
        """One transformer call on packed tokens; `sigmas` in (0, 1) is the model's timestep conditioning."""
        import torch

        transformer = self._require_transformer()
        grid = RESOLUTION // VAE_SCALE
        img_ids = latent_image_ids(grid, grid, self.device, self.dtype)
        txt_ids = torch.zeros(embeds.shape[1], 3, device=self.device, dtype=self.dtype)
        return transformer(
            hidden_states=noisy_tokens,
            timestep=sigmas,
            guidance=None,
            pooled_projections=pooled,
            encoder_hidden_states=embeds,
            txt_ids=txt_ids,
            img_ids=img_ids,
            return_dict=False,
        )[0]

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, seed: int = 0, batch_size: int = 2) -> dict[str, Any]:
        """Held-out flow-matching MSE: every record is VAE-encoded, mixed with a seeded noise tensor at each of
        EVAL_SIGMAS (x_σ = (1 − σ)·x₀ + σ·ε), and the transformer's velocity prediction is scored against the
        rectified-flow target ε − x₀. The same seed gives the same latents, noise and noise levels for the frozen
        and the adapted model, so the numbers are paired."""
        import torch

        checked = validate_dataset(records, min_records=1)["records"]
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        for record in checked:
            self._embeds(record["caption"])
        transformer = self._require_transformer()
        started = time.perf_counter()
        per_sigma: dict[float, list[float]] = {s: [] for s in EVAL_SIGMAS}
        per_record: dict[str, float] = {}
        transformer.eval()
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            latents = self._latents(batch, seed=seed + start)
            tokens = pack_latents(latents)
            embeds, pooled = self._batch_embeds([r["caption"] for r in batch])
            record_losses = [0.0] * len(batch)
            for sigma in EVAL_SIGMAS:
                generator = torch.Generator(device="cpu").manual_seed(seed * 1_000 + int(sigma * 1_000) + start)
                noise = torch.randn(tokens.shape, generator=generator).to(self.device, self.dtype)
                sigmas = torch.full((len(batch),), sigma, device=self.device, dtype=self.dtype)
                noisy = ((1.0 - sigma) * tokens.float() + sigma * noise.float()).to(self.dtype)
                with torch.inference_mode(), self._autocast():
                    pred = self._predict_velocity(noisy, sigmas, embeds, pooled)
                target = noise.float() - tokens.float()
                loss = ((pred.float() - target) ** 2).mean(dim=(1, 2))
                for i, value in enumerate(loss.tolist()):
                    per_sigma[sigma].append(value)
                    record_losses[i] += value / len(EVAL_SIGMAS)
            for record, value in zip(batch, record_losses, strict=True):
                per_record[record["id"]] = round(value, 6)
        by_sigma = {str(s): round(sum(v) / len(v), 6) for s, v in per_sigma.items()}
        mean = sum(per_record.values()) / len(per_record)
        return {
            "metric": "flow_matching_mse (velocity-prediction MSE over the packed latent, mean over records and EVAL_SIGMAS)",
            "n_records": len(checked),
            "sigmas": list(EVAL_SIGMAS),
            "seed": seed,
            "flow_matching_mse": round(mean, 6),
            "by_sigma": by_sigma,
            "per_record": per_record,
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 2),
        }

    # ---- adaptation ------------------------------------------------------------------------------------

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 1e-4,
        batch_size: int = 1,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded QLoRA fine-tuning on the rectified-flow objective: every step draws one noise level per image
        uniformly from (0, 1) and one noise tensor (both seeded), mixes x_σ = (1 − σ)·x₀ + σ·ε, and minimises the
        MSE between the transformer's velocity prediction and ε − x₀. AdamW at a fixed learning rate on the 380 LoRA
        tensors only, over the frozen 4-bit base, with gradient checkpointing and autocast (loss scaling in
        float16). Epoch 0 records the frozen model; the epoch with the lowest validation loss is kept."""
        if not self.use_lora:
            raise ValueError("adapt() needs a pipeline built with use_lora=True")
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 4:
            raise ValueError("batch_size must be an int in 1..4")
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        for record in train_checked + (val_checked or []):
            self._embeds(record["caption"])
        import torch

        model = self._require_transformer()
        torch.manual_seed(seed)
        started = time.perf_counter()
        names = lora_parameter_names(model)
        name_set = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in name_set)
        params = [p for n, p in model.named_parameters() if n in name_set]
        n_trainable = sum(p.numel() for p in params)
        if n_trainable != LORA_PARAMETERS:
            raise ValueError(f"{n_trainable} trainable parameters, expected {LORA_PARAMETERS}")
        initial_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        use_scaler = self.dtype == torch.float16
        scaler = torch.amp.GradScaler(self.device.split(":")[0], enabled=use_scaler)
        generator = torch.Generator(device="cpu").manual_seed(seed)
        # Latents are encoded once (the VAE posterior sample is seeded), so epochs differ only in noise and σ.
        tokens_by_id = {}
        for start in range(0, len(train_checked), 4):
            batch = train_checked[start : start + 4]
            encoded = pack_latents(self._latents(batch, seed=seed + 10_000 + start))
            for record, tokens in zip(batch, encoded, strict=True):
                tokens_by_id[record["id"]] = tokens
        model.enable_gradient_checkpointing()
        try:
            history: list[dict[str, Any]] = []
            entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "note": "frozen model (LoRA at initialisation: B = 0)"}
            entry["val_loss"] = self.evaluate(val_checked, seed=seed)["flow_matching_mse"] if val_checked else None
            history.append(entry)
            if progress:
                progress(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            best_epoch = 0
            n_steps = 0
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    tokens = torch.stack([tokens_by_id[r["id"]] for r in batch])
                    embeds, pooled = self._batch_embeds([r["caption"] for r in batch])
                    noise = torch.randn(tokens.shape, generator=generator).to(self.device, self.dtype)
                    sigmas = torch.rand((len(batch),), generator=generator).to(self.device, self.dtype)
                    mix = sigmas.float().view(-1, 1, 1)
                    noisy = ((1.0 - mix) * tokens.float() + mix * noise.float()).to(self.dtype)
                    with self._autocast():
                        pred = self._predict_velocity(noisy, sigmas, embeds, pooled)
                    loss = torch.nn.functional.mse_loss(pred.float(), noise.float() - tokens.float())
                    optimiser.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimiser)
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(optimiser)
                    scaler.update()
                    losses.append(float(loss.detach()))
                    n_steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses)}
                entry["val_loss"] = self.evaluate(val_checked, seed=seed)["flow_matching_mse"] if val_checked else None
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
                    best_epoch = epoch
        except BaseException:
            # Transactional: any failure leaves the transformer as it was before adapt() — LoRA restored to its
            # initial values, everything frozen, no adapter attached.
            self._write_lora_state(model, initial_state)
            model.disable_gradient_checkpointing()
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        self._write_lora_state(model, best_state)
        model.disable_gradient_checkpointing()
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "method": "QLoRA (peft LoRA over a bitsandbytes NF4 base)",
            "rank": LORA_RANK,
            "alpha": LORA_ALPHA,
            "targets": list(LORA_TARGETS),
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": count_parameters(model),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "optimizer": "AdamW (weight_decay 0, grad-norm clip 1.0)",
            "precision": (
                f"{QUANTIZATION} base, {self.compute_dtype} autocast"
                + (" + GradScaler" if use_scaler else "")
                + ", float32 LoRA masters"
            ),
            "objective": "rectified-flow velocity MSE, uniform σ in (0, 1)",
            "quantization": QUANTIZATION,
            "compute_dtype": self.compute_dtype,
            "n_train_records": len(train_checked),
            "n_steps": n_steps,
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    @staticmethod
    def _write_lora_state(model: Any, values: Mapping[str, Any]) -> None:
        """Overwrite exactly the LoRA tensors in place (a full `load_state_dict` would try to re-quantise the base)."""
        import torch

        params = dict(model.named_parameters())
        with torch.no_grad():
            for name, value in values.items():
                params[name].copy_(value.to(params[name].device, params[name].dtype))

    # ---- artifacts -------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained LoRA tensors as safetensors with a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {
            k: v.detach().to("cpu", dtype=v.dtype).contiguous() for k, v in self.transformer.named_parameters() if k in names
        }
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "license": MODEL_LICENSE,
                "staging": {"repo": STAGING_ID, "revision": STAGING_REVISION},
                "quantization": QUANTIZATION,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    @staticmethod
    def check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> Path:
        """Static checks before any weights work: format and version, the pinned base snapshot and quantisation,
        exactly one weights entry named `adapter.safetensors` inside the artifact directory, and the pinned LoRA
        configuration. Returns the weights path."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("quantization") != QUANTIZATION:
            raise ValueError(
                f"artifact was trained over a {base.get('quantization')!r} base, this pipeline uses {QUANTIZATION!r}"
            )
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        declared = None
        if isinstance(adapter, Mapping):
            declared = (adapter.get("rank"), adapter.get("alpha"), list(adapter.get("targets", [])))
        if declared != (LORA_RANK, LORA_ALPHA, list(LORA_TARGETS)):
            raise ValueError(
                f"artifact adapter must declare rank {LORA_RANK}, alpha {LORA_ALPHA} and targets {list(LORA_TARGETS)}"
            )
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the LoRA tensors."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self.check_artifact_manifest(root, manifest)
        if not self.use_lora:
            raise ValueError("this adapter carries LoRA tensors; build the pipeline with use_lora=True")
        model = self._require_transformer()
        expected = lora_parameter_names(model)
        if sorted(manifest["tensors"]) != expected:
            raise ValueError(f"artifact tensor list does not match the {len(expected)} LoRA tensors of this model")
        entry = manifest["files"][0]
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        params = dict(model.named_parameters())
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(params[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, model has {tuple(params[key].shape)}")
        self._write_lora_state(model, tensors)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        compute_dtype: str = COMPUTE_DTYPE,
        prompt_cache: Mapping[str, Mapping[str, Any]] | None = None,
    ) -> FluxSchnellPipeline:
        """Fresh pipeline with the adapter applied. `prompt_cache` (from `export_prompt_cache`) lets the reload skip
        the text encoders; without it, encode prompts before the first generate/evaluate call."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        cls.check_artifact_manifest(root, manifest)
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, use_lora=True, compute_dtype=compute_dtype
        )
        if prompt_cache:
            pipeline.import_prompt_cache(prompt_cache)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/flux_schnell_generation_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Evaluation of generated images with a frozen CLIP scorer: prompt alignment, zero-shot label accuracy among
the dataset's captions, and similarity to the held-out real photographs of the same caption.

The scorer is the pinned `laion/CLIP-ViT-B-32-laion2B-s34B-b79K` snapshot (MIT), loaded from its verified
directory; it never takes part in generation or training, so a change in these scores can only come from the
generator. Every number is a cosine similarity of L2-normalised CLIP embeddings (×100) or an accuracy derived
from one — tutorial sample-sanity evidence, not a benchmark, and not a human judgement of image quality.
"""

from __future__ import annotations

import time
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import SCORER_ID, SCORER_REVISION, SCORER_WEIGHTS_DIR, stage_missing_scorer_files, verify_scorer_snapshot` removed — names are kernel globals defined by the carried modules


def _projected(features: Any) -> Any:
    """The projected CLIP embedding as a tensor: `transformers` 5 returns the encoder's model output (the projection
    written into `pooler_output`) from `get_text_features` / `get_image_features`, earlier releases the tensor."""
    return features if hasattr(features, "shape") else features.pooler_output


class ClipScorer:
    """Frozen CLIP image/text embedder on a verified snapshot."""

    def __init__(self, *, device: str | None = None, weights_dir: str | Path | None = None, allow_download: bool = False) -> None:
        root = Path(weights_dir) if weights_dir is not None else SCORER_WEIGHTS_DIR
        stage_missing_scorer_files(root, allow_download=allow_download)
        verify_scorer_snapshot(root)
        import torch
        from transformers import CLIPModel, CLIPProcessor

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(str(root), torch_dtype=torch.float32).to(self.device).eval()
        for param in self.model.parameters():
            param.requires_grad_(False)
        self.processor = CLIPProcessor.from_pretrained(str(root))
        self.weights_dir = root
        self.identity = {"id": SCORER_ID, "revision": SCORER_REVISION}

    def image_embeddings(self, images: Sequence[Any]) -> Any:
        import torch

        out = []
        with torch.inference_mode():
            for start in range(0, len(images), 16):
                batch = self.processor(images=[im.convert("RGB") for im in images[start : start + 16]], return_tensors="pt")
                features = _projected(self.model.get_image_features(pixel_values=batch["pixel_values"].to(self.device)))
                out.append(torch.nn.functional.normalize(features.float(), dim=-1).cpu())
        return torch.cat(out)

    def text_embeddings(self, texts: Sequence[str]) -> Any:
        import torch

        with torch.inference_mode():
            batch = self.processor(text=list(texts), return_tensors="pt", padding=True, truncation=True)
            features = _projected(
                self.model.get_text_features(
                    input_ids=batch["input_ids"].to(self.device), attention_mask=batch["attention_mask"].to(self.device)
                )
            )
        return torch.nn.functional.normalize(features.float(), dim=-1).cpu()


def score_generations(
    scorer: ClipScorer,
    generated: Sequence[Mapping[str, Any]],
    *,
    references: Sequence[Mapping[str, Any]] | None = None,
) -> dict[str, Any]:
    """Score `generated` records (`{prompt, image}`) with three measures:

    * `clip_prompt_similarity` — mean cosine(image, its prompt) × 100 (prompt alignment);
    * `label_accuracy` — the fraction of images whose nearest prompt among the distinct prompts is their own
      (zero-shot classification of the generated image among the dataset's captions; argmax rule);
    * `reference_similarity` — mean cosine(image, mean embedding of the real `references` with the same caption)
      × 100 when references are given (how close the generations sit to the held-out photographs)."""
    if not generated:
        raise ValueError("no generated records to score")
    started = time.perf_counter()
    prompts = list(dict.fromkeys(str(g["prompt"]) for g in generated))
    text = scorer.text_embeddings(prompts)
    images = scorer.image_embeddings([g["image"] for g in generated])
    index = {p: i for i, p in enumerate(prompts)}
    sims = images @ text.T  # (n_images, n_prompts)
    own = [float(sims[i, index[str(g["prompt"])]]) for i, g in enumerate(generated)]
    nearest = sims.argmax(dim=1).tolist()
    correct = [nearest[i] == index[str(g["prompt"])] for i, g in enumerate(generated)]
    per_image = []
    ref_means: dict[str, Any] = {}
    if references:
        ref_images = scorer.image_embeddings([r["image"] for r in references])
        for caption in prompts:
            rows = [i for i, r in enumerate(references) if str(r["caption"]) == caption]
            if rows:
                mean = ref_images[rows].mean(dim=0)
                ref_means[caption] = mean / mean.norm()
    ref_scores = []
    for i, g in enumerate(generated):
        entry = {
            "prompt": str(g["prompt"]),
            "seed": g.get("seed"),
            "clip_prompt_similarity": round(own[i] * 100, 3),
            "nearest_prompt": prompts[nearest[i]],
            "correct": bool(correct[i]),
        }
        if str(g["prompt"]) in ref_means:
            value = float(images[i] @ ref_means[str(g["prompt"])]) * 100
            entry["reference_similarity"] = round(value, 3)
            ref_scores.append(value)
        per_image.append(entry)
    report: dict[str, Any] = {
        "scorer": dict(scorer.identity),
        "n_images": len(generated),
        "n_prompts": len(prompts),
        "clip_prompt_similarity": round(sum(own) / len(own) * 100, 3),
        "label_accuracy": round(sum(correct) / len(correct), 4),
        "decision_rule": "argmax cosine similarity over the distinct prompts (no threshold)",
        "per_image": per_image,
        "seconds": round(time.perf_counter() - started, 2),
    }
    if ref_scores:
        report["reference_similarity"] = round(sum(ref_scores) / len(ref_scores), 3)
        report["n_references"] = len(references or [])
    return report


def real_photo_baseline(scorer: ClipScorer, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """The same three measures on real photographs of the dataset (image = the record's own photo, prompt = its
    caption, references = the other records): the ceiling a generator could reach on these metrics."""
    generated = [{"prompt": r["caption"], "image": r["image"], "seed": None} for r in records]
    report = score_generations(scorer, generated, references=records)
    report["note"] = "real held-out photographs scored as if generated (references include each photo itself)"
    return report

**Module 3/3:** `src/flux_schnell_generation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Captioned-image dataset contract for adapting the generator: the pinned iNaturalist bird sample, seeded
splitting, generation prompts, BYOD loaders and CSV export.

The default dataset is **real** and narrow on purpose: 60 CC0-licensed, research-grade iNaturalist photographs of
six common North American birds (10 per species, one per observer per species), a subset of the corpus the
fleet's DINOv2 and ViT rows pinned on 2026-09-19, pinned here by photo id, byte size and SHA-256 of the served
`medium` JPEG (about 500 px on the longer side). Every file is fetched from the iNaturalist open-data bucket at
run time and refused on any byte-size or SHA-256 mismatch; the repository redistributes none of the photographs.
Each record keeps the observation id and observer login so every image is traceable to its public observation
page. Captions are generated from the species names by one template, so the adaptation teaches the generator
what these six names look like in this kind of photograph.

A record is ``{id, image, caption}``: a PIL image (or a path to one) and the caption used to generate it.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MIN_TRAIN_RECORDS, MODEL_ID, image_digest, validate_dataset` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species, 10 each)"
CORPUS_RELEASE = "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (fleet DINOv2/ViT corpus subset)"
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = "CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)"
CORPUS_BYTES = 5_789_324
DEFAULT_CACHE_DIR = Path("weights") / "inat-birds"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 6, "validation": 2, "test": 2}  # per species; 6 species -> 36 / 12 / 12
CAPTION_TEMPLATE = "a photo of a {common_name} ({scientific_name}), a wild bird photographed outdoors"
SPECIES: dict[str, tuple[str, str]] = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
# (id, label, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served
#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension
#  (jpg or jpeg); the digest pins the served bytes
SAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (
    (
        "song_sparrow-00",
        "song_sparrow",
        129376982,
        79016324,
        "andywilson",
        43427,
        "7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d",
        "jpg",
    ),
    (
        "song_sparrow-01",
        "song_sparrow",
        480991086,
        267636534,
        "lyneisfilm",
        162073,
        "11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec",
        "jpg",
    ),
    (
        "song_sparrow-02",
        "song_sparrow",
        546060381,
        302980489,
        "swpollinators",
        27899,
        "4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6",
        "jpg",
    ),
    (
        "song_sparrow-03",
        "song_sparrow",
        308625896,
        177450028,
        "radrat",
        70961,
        "1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5",
        "jpeg",
    ),
    (
        "song_sparrow-04",
        "song_sparrow",
        494793016,
        275349085,
        "k-simpkins",
        58410,
        "255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e",
        "jpg",
    ),
    (
        "song_sparrow-05",
        "song_sparrow",
        674054489,
        369029444,
        "ben142",
        220573,
        "1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123",
        "jpg",
    ),
    (
        "song_sparrow-06",
        "song_sparrow",
        339623726,
        193339933,
        "rawcomposition",
        25012,
        "d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b",
        "jpg",
    ),
    (
        "song_sparrow-07",
        "song_sparrow",
        222768957,
        130949329,
        "davidfbird",
        110773,
        "0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff",
        "jpg",
    ),
    (
        "song_sparrow-08",
        "song_sparrow",
        181658744,
        107953669,
        "gcart043",
        98482,
        "a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4",
        "jpeg",
    ),
    (
        "song_sparrow-09",
        "song_sparrow",
        148994242,
        90171417,
        "glennberry",
        102702,
        "64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa",
        "jpg",
    ),
    (
        "chipping_sparrow-00",
        "chipping_sparrow",
        198992636,
        117809422,
        "k-simpkins",
        62563,
        "cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf",
        "jpg",
    ),
    (
        "chipping_sparrow-01",
        "chipping_sparrow",
        248210057,
        144599194,
        "w_mark_c",
        193389,
        "497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150",
        "jpg",
    ),
    (
        "chipping_sparrow-02",
        "chipping_sparrow",
        156350853,
        94266719,
        "ellyne",
        142332,
        "6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2",
        "jpeg",
    ),
    (
        "chipping_sparrow-03",
        "chipping_sparrow",
        16128796,
        11327134,
        "reuvenm",
        70532,
        "5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c",
        "jpeg",
    ),
    (
        "chipping_sparrow-04",
        "chipping_sparrow",
        391300648,
        220982684,
        "carterdorscht",
        208567,
        "c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21",
        "jpeg",
    ),
    (
        "chipping_sparrow-05",
        "chipping_sparrow",
        339456083,
        193252042,
        "rawcomposition",
        46329,
        "41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496",
        "jpg",
    ),
    (
        "chipping_sparrow-06",
        "chipping_sparrow",
        40457652,
        26076708,
        "andywilson",
        156894,
        "b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6",
        "jpeg",
    ),
    (
        "chipping_sparrow-07",
        "chipping_sparrow",
        84151914,
        52921135,
        "davidfbird",
        145714,
        "ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9",
        "jpeg",
    ),
    (
        "chipping_sparrow-08",
        "chipping_sparrow",
        523674610,
        291074747,
        "rwp84",
        47983,
        "41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800",
        "jpg",
    ),
    (
        "chipping_sparrow-09",
        "chipping_sparrow",
        292695018,
        168861389,
        "tim_kirsten",
        64812,
        "cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d",
        "jpeg",
    ),
    (
        "white_throated_sparrow-00",
        "white_throated_sparrow",
        339621218,
        193338380,
        "rawcomposition",
        31152,
        "d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6",
        "jpg",
    ),
    (
        "white_throated_sparrow-01",
        "white_throated_sparrow",
        166821399,
        99992799,
        "dziakj1",
        125954,
        "c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852",
        "jpeg",
    ),
    (
        "white_throated_sparrow-02",
        "white_throated_sparrow",
        469820434,
        261505977,
        "joy4birds",
        111767,
        "7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb",
        "jpg",
    ),
    (
        "white_throated_sparrow-03",
        "white_throated_sparrow",
        99351488,
        62040646,
        "bradenjudson",
        23399,
        "e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e",
        "jpeg",
    ),
    (
        "white_throated_sparrow-04",
        "white_throated_sparrow",
        177497964,
        105746665,
        "andywilson",
        29827,
        "65475d4842396f2488167453192d4aa834d2f1b40bcc78b420878c55ccf694d7",
        "jpeg",
    ),
    (
        "white_throated_sparrow-05",
        "white_throated_sparrow",
        628148203,
        344731686,
        "lavenderdame",
        106872,
        "36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8",
        "jpg",
    ),
    (
        "white_throated_sparrow-06",
        "white_throated_sparrow",
        104660609,
        65043951,
        "allan7",
        42443,
        "10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635",
        "jpeg",
    ),
    (
        "white_throated_sparrow-07",
        "white_throated_sparrow",
        250718938,
        145903421,
        "stevestevens",
        138735,
        "bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-08",
        "white_throated_sparrow",
        15105971,
        10793852,
        "schylerbrown",
        31467,
        "6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-09",
        "white_throated_sparrow",
        171460784,
        102554447,
        "w_mark_c",
        251287,
        "3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137",
        "jpg",
    ),
    (
        "dark_eyed_junco-00",
        "dark_eyed_junco",
        172110799,
        102901486,
        "schylerbrown",
        182973,
        "185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9",
        "jpeg",
    ),
    (
        "dark_eyed_junco-01",
        "dark_eyed_junco",
        46691943,
        29901256,
        "haida_gwaii",
        46823,
        "a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5",
        "jpg",
    ),
    (
        "dark_eyed_junco-02",
        "dark_eyed_junco",
        707222551,
        386266764,
        "ben142",
        289608,
        "7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89",
        "jpg",
    ),
    (
        "dark_eyed_junco-03",
        "dark_eyed_junco",
        192557376,
        114006980,
        "k-simpkins",
        172398,
        "206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479",
        "jpeg",
    ),
    (
        "dark_eyed_junco-04",
        "dark_eyed_junco",
        8793471,
        6892999,
        "truthseqr",
        45302,
        "f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2",
        "jpeg",
    ),
    (
        "dark_eyed_junco-05",
        "dark_eyed_junco",
        346777340,
        196961623,
        "zacharyfoster",
        46712,
        "ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab",
        "jpg",
    ),
    (
        "dark_eyed_junco-06",
        "dark_eyed_junco",
        274980085,
        159160633,
        "andy71",
        51209,
        "c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b",
        "jpeg",
    ),
    (
        "dark_eyed_junco-07",
        "dark_eyed_junco",
        243303909,
        141959574,
        "andywilson",
        71321,
        "ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6",
        "jpeg",
    ),
    (
        "dark_eyed_junco-08",
        "dark_eyed_junco",
        213798376,
        126031618,
        "nathanael15",
        57646,
        "d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b",
        "jpg",
    ),
    (
        "dark_eyed_junco-09",
        "dark_eyed_junco",
        63482066,
        39977347,
        "chrisleearm",
        41870,
        "90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72",
        "jpeg",
    ),
    (
        "house_finch-00",
        "house_finch",
        117990649,
        72375345,
        "kristen163",
        75945,
        "eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f",
        "jpeg",
    ),
    (
        "house_finch-01",
        "house_finch",
        389479656,
        220010434,
        "aster-asti",
        82128,
        "a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5",
        "jpg",
    ),
    (
        "house_finch-02",
        "house_finch",
        176982307,
        105476125,
        "vicki936",
        22211,
        "c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f",
        "jpeg",
    ),
    (
        "house_finch-03",
        "house_finch",
        697940852,
        381438133,
        "ben142",
        196735,
        "a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5",
        "jpg",
    ),
    (
        "house_finch-04",
        "house_finch",
        72470599,
        45698380,
        "henrya",
        61724,
        "1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b",
        "jpeg",
    ),
    (
        "house_finch-05",
        "house_finch",
        98576538,
        61594129,
        "enspring",
        46784,
        "8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c",
        "jpg",
    ),
    (
        "house_finch-06",
        "house_finch",
        80751781,
        50842166,
        "leahmfulton",
        44269,
        "6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7",
        "jpg",
    ),
    (
        "house_finch-07",
        "house_finch",
        630196420,
        345777550,
        "truthseqr",
        149455,
        "abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c",
        "jpg",
    ),
    (
        "house_finch-08",
        "house_finch",
        214612538,
        126483167,
        "hamiltonturner",
        124116,
        "6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930",
        "jpeg",
    ),
    (
        "house_finch-09",
        "house_finch",
        213077180,
        125637342,
        "jnicat",
        25823,
        "e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8",
        "jpeg",
    ),
    (
        "american_goldfinch-00",
        "american_goldfinch",
        84579952,
        53187208,
        "glennberry",
        59673,
        "72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36",
        "jpeg",
    ),
    (
        "american_goldfinch-01",
        "american_goldfinch",
        12533322,
        9255418,
        "braincellsgone",
        55661,
        "6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc",
        "jpg",
    ),
    (
        "american_goldfinch-02",
        "american_goldfinch",
        131102823,
        80016788,
        "radrat",
        98990,
        "232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff",
        "jpeg",
    ),
    (
        "american_goldfinch-03",
        "american_goldfinch",
        175048222,
        104466897,
        "eug302",
        44231,
        "11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e",
        "jpg",
    ),
    (
        "american_goldfinch-04",
        "american_goldfinch",
        68849595,
        43390778,
        "mefisher",
        154503,
        "66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1",
        "jpg",
    ),
    (
        "american_goldfinch-05",
        "american_goldfinch",
        431916465,
        242278180,
        "k-simpkins",
        45413,
        "d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d",
        "jpg",
    ),
    (
        "american_goldfinch-06",
        "american_goldfinch",
        377136648,
        213398931,
        "nathan1177",
        66516,
        "7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438",
        "jpg",
    ),
    (
        "american_goldfinch-07",
        "american_goldfinch",
        230801384,
        135330401,
        "enspring",
        42886,
        "78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed",
        "jpeg",
    ),
    (
        "american_goldfinch-08",
        "american_goldfinch",
        660472044,
        361884286,
        "ben142",
        270599,
        "733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876",
        "jpg",
    ),
    (
        "american_goldfinch-09",
        "american_goldfinch",
        294667312,
        169935316,
        "dande",
        163470,
        "39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0",
        "jpeg",
    ),
)
SAMPLE_LABEL_SOURCE = f"{CORPUS_NAME}; {CORPUS_RELEASE}; {CORPUS_LICENSE}"


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def caption_for(label: str) -> str:
    """The template caption of a species key (the prompt the tutorial trains and generates with)."""
    if label not in SPECIES:
        raise ValueError(f"unknown species key {label!r}; expected one of {sorted(SPECIES)}")
    scientific, common = SPECIES[label]
    return CAPTION_TEMPLATE.format(common_name=common, scientific_name=scientific)


def photo_url(photo_id: int, ext: str = "jpg") -> str:
    """The served object for a pinned photo; `ext` is its recorded original extension (jpg or jpeg)."""
    if ext not in ("jpg", "jpeg"):
        raise ValueError(f"unsupported photo extension {ext!r}")
    return f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"


def observation_url(observation_id: int) -> str:
    return f"https://www.inaturalist.org/observations/{observation_id}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:
        local = cache / f"{photo_id}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(photo_id, ext)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(url, headers={"User-Agent": "dimer-flux-schnell-tutorial/1.0"})
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 "
                    f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photo bytes into `{id, image, caption, label}` records with their provenance."""
    from PIL import Image

    out = []
    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "caption": caption_for(label),
                "label": label,
                "scientific_name": SPECIES[label][0],
                "common_name": SPECIES[label][1],
                "inat_photo_id": photo_id,
                "inat_observation_url": observation_url(obs_id),
                "observer": user,
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw per species: `sizes` counts per class for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record["label"]), []).append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            out[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    for name in out:
        rng.shuffle(out[name])
        out[name] = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes)


def sample_prompts(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The distinct captions of a split, in first-seen order (the prompts generation and scoring use)."""
    return list(dict.fromkeys(str(r["caption"]) for r in records))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test, grouped by caption, after de-duplicating
    images. Every caption keeps at least one test record when it has three or more images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_caption: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_caption.setdefault(record["caption"], []).append(record)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for caption in sorted(by_caption):
        pool = by_caption[caption]
        rng.shuffle(pool)
        n_test = max(1, round(len(pool) * test_fraction)) if len(pool) >= 3 else 0
        n_val = round(len(pool) * val_fraction) if len(pool) >= 3 else 0
        splits["test"].extend(pool[:n_test])
        splits["validation"].extend(pool[n_test : n_test + n_val])
        splits["train"].extend(pool[n_test + n_val :])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_TRAIN_RECORDS:
        raise ValueError(f"split leaves {len(splits['train'])} training records; at least {MIN_TRAIN_RECORDS} are required")
    if not splits["test"]:
        raise ValueError("split leaves no test record; give at least one caption three or more images")
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, caption}` records from a directory or a zip holding `captions.csv` (columns `id`, `file`,
    `caption`) beside the image files; images are decoded, never extracted to disk."""
    from PIL import Image

    source = Path(path)
    if source.is_dir():
        table = (source / "captions.csv").read_text(encoding="utf-8")
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "captions.csv" not in members:
            raise ValueError("BYOD zip must contain captions.csv")
        table = archive.read(members["captions.csv"]).decode("utf-8")
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding captions.csv and the image files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "file", "caption"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"captions.csv is missing columns {sorted(missing)}")
    out = []
    for row in rows:
        image = loader(row["file"])
        image.load()
        out.append({"id": row["id"], "image": image.convert("RGB"), "caption": row["caption"]})
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the captions table of a split (id, file, caption, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "file", "caption", "label", "observer", "inat_observation_url"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "file": f"{record['inat_photo_id']}.jpg" if record.get("inat_photo_id") else f"{record['id']}.jpg",
                    "caption": record["caption"],
                    "label": record.get("label", ""),
                    "observer": record.get("observer", ""),
                    "inat_observation_url": record.get("inat_observation_url", ""),
                }
            )
    return out


def dataset_manifest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Validate every split and summarise the dataset (counts, captions, digests) for provenance exports."""
    summary: dict[str, Any] = {"model_id": MODEL_ID, "splits": {}}
    for name, records in splits.items():
        report = validate_dataset(records, min_records=1)
        summary["splits"][name] = {
            "n_records": report["n_records"],
            "n_captions": report["n_captions"],
            "shorter_side": report["shorter_side"],
            "centre_cropped": report["centre_cropped"],
            "digest": report["digest"],
        }
    summary["disjoint"] = check_split_disjoint(splits)
    digests = json.dumps({k: v["digest"] for k, v in summary["splits"].items()}, sort_keys=True)
    summary["digest"] = hashlib.sha256(digests.encode("utf-8")).hexdigest()
    return summary

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `23`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub (the upstream identity is `black-forest-labs/FLUX.1-schnell`; the bytes are served by the ungated mirror `unsloth/FLUX.1-schnell` at `9df3faa7…`, see Section 3) **at upstream revision `741f7c3ce8b3…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `FluxSchnellPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, use_lora=True)` load the verified files. The bytes themselves come from the ungated mirror `STAGING_ID` = `unsloth/FLUX.1-schnell` at `STAGING_REVISION` = `9df3faa7…` (`verify_snapshot` checks that the manifest names exactly that source): the upstream repository is click-through gated and hides its LFS digests behind the gate, so what makes the mirror trustworthy here is the manifest — every byte loaded is one whose SHA-256 the repository committed, and the 23 files match the upstream tree file for file and byte for byte in size. There is no fallback to a different download and no remote model code is executed; the model classes come from `diffusers`, `transformers`, `peft` and `bitsandbytes` on PyPI. `from_pretrained` loads the VAE, both tokenizers and the scheduler config and leaves the transformer for Section 6: a 4-bit 12 B transformer (6.6 GB) and the two 16-bit text encoders (9.7 GB) do not fit a 16 GB GPU together, so prompts are encoded first. The package also pins a second snapshot `clip-vit-b-32-laion2b` (9 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "flux1-schnell",
  "modelId": "black-forest-labs/FLUX.1-schnell",
  "revision": "741f7c3ce8b383c54771c7003378a50191e9efe9",
  "license": "apache-2.0",
  "staging": {
    "repo": "unsloth/FLUX.1-schnell",
    "revision": "9df3faa7ae3b6ddf0b2b69bb78616372897cc65c",
    "note": "ungated, unmodified mirror; every listed path exists in the upstream tree with the same byte size (checked 2026-09-20 through the ungated tree API); upstream LFS digests are hidden behind the gate"
  },
  "files": [
    {
      "path": "model_index.json",
      "bytes": 536,
      "sha256": "24946df21ff25e210486b5f6b14208983a90c9c73f8d48cfa724c0e4e03f7201"
    },
    {
      "path": "scheduler/scheduler_config.json",
      "bytes": 274,
      "sha256": "b129cebacf8f851867ec5c7c4d3f4bf787e232525a53becf4df5a72278a788d5"
    },
    {
      "path": "text_encoder/config.json",
      "bytes": 613,
      "sha256": "d79d5c8c6ce85112a923d621a5412886ddbbb0636210fc0f72f450582e675542"
    },
    {
      "path": "text_encoder/model.safetensors",
      "bytes": 246144352,
      "sha256": "893d67a23f4693ed42cdab4cbad7fe3e727cf59609c40da28a46b5470f9ed082"
    },
    {
      "path": "text_encoder_2/config.json",
      "bytes": 782,
      "sha256": "9001e5a8ae0571a362f806b87b6105dd1a15c33dca237b606d2561164109beeb"
    },
    {
      "path": "text_encoder_2/model-00001-of-00002.safetensors",
      "bytes": 4994582224,
      "sha256": "ec87bffd1923e8b2774a6d240c922a41f6143081d52cf83b8fe39e9d838c893e"
    },
    {
      "path": "text_encoder_2/model-00002-of-00002.safetensors",
      "bytes": 4530066360,
      "sha256": "a5640855b301fcdbceddfa90ae8066cd9414aff020552a201a255ecf2059da00"
    },
    {
      "path": "text_encoder_2/model.safetensors.index.json",
      "bytes": 19885,
      "sha256": "3bacec0f0cf392399d4a385908f67dd73df99c9e9cfee669f148858ba9fbdb0a"
    },
    {
      "path": "tokenizer/merges.txt",
      "bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "path": "tokenizer/special_tokens_map.json",
      "bytes": 588,
      "sha256": "2cdb3b8331a60c92fc1e55a13e9fd61fd2293c5a51275fdcccd62b780052530e"
    },
    {
      "path": "tokenizer/tokenizer_config.json",
      "bytes": 705,
      "sha256": "6bdcee9ccce2a16ca2b4c0c5ed00b42c50ea225f4472a8c4c1e963a2902c2881"
    },
    {
      "path": "tokenizer/vocab.json",
      "bytes": 1059962,
      "sha256": "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"
    },
    {
      "path": "tokenizer_2/special_tokens_map.json",
      "bytes": 2543,
      "sha256": "7a1985a994c41886db38c719d2a3d2f40606663cc19d7c5d6a85d349320e06d2"
    },
    {
      "path": "tokenizer_2/spiece.model",
      "bytes": 791656,
      "sha256": "d60acb128cf7b7f2536e8f38a5b18a05535c9e14c7a355904270e15b0945ea86"
    },
    {
      "path": "tokenizer_2/tokenizer.json",
      "bytes": 2424235,
      "sha256": "f5dfec163765e18e270537fe896c49f5fad74db1525641d9b255a3008b999596"
    },
    {
      "path": "tokenizer_2/tokenizer_config.json",
      "bytes": 20817,
      "sha256": "1a3d2db64215ed77854dd4208aac5f8361c1b5471cabd19c0ef1472d1a895eb0"
    },
    {
      "path": "transformer/config.json",
      "bytes": 321,
      "sha256": "397cfb92299488013ec3af6142a2a877366f8d2e44efbb4f3e33479e7960d3d0"
    },
    {
      "path": "transformer/diffusion_pytorch_model-00001-of-00003.safetensors",
      "bytes": 9962580296,
      "sha256": "9b633dbe87316385c5b1c262bd4b5a01e3d955170661d63dcec8a01e89c0d820"
    },
    {
      "path": "transformer/diffusion_pytorch_model-00002-of-00003.safetensors",
      "bytes": 9949328904,
      "sha256": "58b4434078f0c2567ddc54e3b5cbf39626ab55fbd9d5c22956e183668f535dec"
    },
    {
      "path": "transformer/diffusion_pytorch_model-00003-of-00003.safetensors",
      "bytes": 3870584832,
      "sha256": "e2cbc25471ed5186e69a9b51098300cb2f612556453e38a372c851a220ed238d"
    },
    {
      "path": "transformer/diffusion_pytorch_model.safetensors.index.json",
      "bytes": 120822,
      "sha256": "783f857a5872f069e75daf4a5abe5efd6ff9ec2f37d71159767910cebfe048a6"
    },
    {
      "path": "vae/config.json",
      "bytes": 774,
      "sha256": "bc1e208f414a315365fbecf426838f43b87c9d5c051219e0968a56e2644b2998"
    },
    {
      "path": "vae/diffusion_pytorch_model.safetensors",
      "bytes": 167666902,
      "sha256": "f5b59a26851551b67ae1fe58d32e76486e1e812def4696a4bea97f16604d40a3"
    }
  ],
  "totalBytes": 33725923002
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

SCORER_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "clip-vit-b-32-laion2b",
  "modelId": "laion/CLIP-ViT-B-32-laion2B-s34B-b79K",
  "revision": "1a25a446712ba5ee05982a381eed697ef9b435cf",
  "files": [
    {
      "path": "README.md",
      "bytes": 7464,
      "sha256": "d0600101ba66849bd84eeae14e0b1ccd56cd962ea94aec5d60c415d9ce4e5f28"
    },
    {
      "path": "config.json",
      "bytes": 4355,
      "sha256": "1284cbff35169abb23a1c5525a8b0f543c7bd191d4b9aed63880c1571bc4191c"
    },
    {
      "path": "merges.txt",
      "bytes": 524618,
      "sha256": "f1cd414c2d113a28b296afc0fd66ae854c0e0c34d5de373a2f5b7f62e0c02344"
    },
    {
      "path": "model.safetensors",
      "bytes": 605157884,
      "sha256": "74813fbcdc750f235c9784c367ca1394d2a5c25eb0aac92761752ac239db7cff"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 316,
      "sha256": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 389,
      "sha256": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2224041,
      "sha256": "b556ac8c99757ffb677208af34bc8c6721572114111a6e0aaf5fa69ff0b8d842"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 904,
      "sha256": "e19f34ef773563fb695f96cfcae1e4c7b112ab6ad532f6962061df5242d924f0"
    },
    {
      "path": "vocab.json",
      "bytes": 862328,
      "sha256": "5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363"
    }
  ],
  "totalBytes": 608782299
}

if (SCORER_MANIFEST['modelId'], SCORER_MANIFEST['revision']) != (SCORER_ID, SCORER_REVISION):
    raise RuntimeError('inline clip-vit-b-32-laion2b manifest does not name the identity carried by the pipeline module')
SCORER_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(SCORER_WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(SCORER_MANIFEST, handle, indent=2)
fetched_clip_vit_b_32_laion2b = stage_missing_scorer_files(SCORER_WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(SCORER_WEIGHTS_DIR), 'fetched': fetched_clip_vit_b_32_laion2b})
_extra = verify_scorer_snapshot(SCORER_WEIGHTS_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_clip_vit_b_32_laion2b': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = FluxSchnellPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, use_lora=True)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample photographs, validation and splits

The default dataset is 60 research-grade iNaturalist photographs of six common North American birds — 10 per species, one per observer per species, every one CC0 — fetched by photo id from the open-data bucket and refused on any byte-size or SHA-256 mismatch (`fetch_corpus`). Each photo's caption is generated from its species by one template, so the adaptation teaches the generator what six names look like in this kind of photograph. `build_sample_dataset` draws a seeded stratified split — 6 / 2 / 2 per species for training, validation and test — and `dataset_manifest` validates every split, checks that no image appears twice and records a digest.

Look for: 36 / 12 / 12 records, six distinct captions, a shorter side around 300..500 px (every photo is centre-cropped to 512²), a written `outputs/flux_schnell_generation_sample_captions.csv` in the shape BYOD expects, and three refusal probes — a missing caption, a 200 px image, a duplicate id — each rejected before the model runs.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    splits = split_dataset(load_byod_dataset(byod_path), seed=0)
    data_source = 'BYOD (' + file_name + ')'
else:
    splits = fetch_sample_dataset(cache_dir='weights/inat-birds')
    data_source = SAMPLE_LABEL_SOURCE
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']

dataset_report = dataset_manifest({'train': train_records, 'validation': val_records, 'test': test_records})
print({'data_source': data_source, 'splits': {k: v['n_records'] for k, v in dataset_report['splits'].items()}, 'captions': dataset_report['splits']['train']['n_captions'], 'disjoint': dataset_report['disjoint']})
print({'shorter_side': dataset_report['splits']['train']['shorter_side'], 'centre_cropped': dataset_report['splits']['train']['centre_cropped'], 'digest': dataset_report['digest'][:16] + '...'})
print({'first_test_record': validate_inputs(test_records[0]), 'caption': test_records[0]['caption']})
prompts = sample_prompts(train_records)
print({'prompts': prompts})
sample_csv = write_dataset_csv(test_records, 'outputs/flux_schnell_generation_sample_captions.csv')
print({'sample_csv': str(sample_csv)})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'missing caption': [{'id': r['id'], 'image': r['image']} for r in train_records[:4]],
    'image too small': [{**train_records[0], 'image': Image.new('RGB', (200, 200))}, *train_records[1:4]],
    'duplicate id': [train_records[0], *train_records[:4]],
}
for name, records in probes.items():
    try:
        validate_dataset(records)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Encode every prompt, then release the text encoders

`pipe.encode_prompts` loads CLIP-L and T5-XXL from the verified snapshot (float16 on CUDA), tokenises each distinct prompt — 77 CLIP tokens for the pooled vector, 256 T5 tokens for the sequence — runs both encoders once per prompt and keeps the (256, 4096) embeddings and the (768,) pooled vector on the CPU. Encoded here: the six training captions (which are also the validation and test captions and the generation prompts) and one new prompt for Section 9. schnell is guidance-distilled, so there is no negative prompt to encode. `release_text_encoder` then drops the 9.7 GB of encoders so the 4-bit transformer, the VAE, the scorer and a training graph fit on a 16 GB GPU. If the transformer is already resident (a BYOD re-run), it is released first — the two never share the GPU.

Look for: the encoders loading in about a minute from the bfloat16 shards, seven prompts encoded in seconds, no truncation, finite embeddings, and the GPU memory falling back to nearly zero after the release.

In [ ]:
import time

NEW_PROMPT = 'a photo of a House Finch (Haemorhous mexicanus) perched on a snow-covered branch in winter'

def gpu_memory_gb():
    return round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None

if pipe.transformer is not None:
    print({'transformer_released_for_encoding': pipe.release_transformer()})
all_prompts = sample_prompts(train_records + val_records + test_records) + [NEW_PROMPT]
encode_report = pipe.encode_prompts(all_prompts)
print({**encode_report, 'gpu_memory_gb_with_encoders': gpu_memory_gb()})
cache = pipe.export_prompt_cache()
print({'embeddings_finite': all(bool(torch.isfinite(v['embeds']).all()) and bool(torch.isfinite(v['pooled']).all()) for v in cache.values()), 'embeds_shape': tuple(next(iter(cache.values()))['embeds'].shape), 'pooled_shape': tuple(next(iter(cache.values()))['pooled'].shape)})
released = pipe.release_text_encoder()
print({'text_encoders_released': released, 'gpu_memory_gb_after_release': gpu_memory_gb(), 'device': pipe.device, 'compute_dtype': pipe.compute_dtype})

## 6. Load the 4-bit transformer; the frozen model's held-out loss and CLIP-scored generations

`pipe.load_transformer` reads the three bfloat16 shards (23.8 GB) and quantises every linear projection of the 57 blocks to 4-bit NF4 with double quantisation as it loads — about 6.6 GB on the GPU with float16 compute — checks the parameter count against the checkpoint (11,891,178,560), freezes everything and attaches the untrained LoRA adapter (`use_lora=True`; its B matrices start at zero, so until Section 7 this is the pretrained model). Two kinds of number are read here and kept for the comparison.

**Held-out flow-matching loss** (`pipe.evaluate`): each held-out photograph is VAE-encoded and packed, mixed with a seeded noise tensor at five fixed noise levels σ ∈ {0.1, 0.3, 0.5, 0.7, 0.9} as x_σ = (1 − σ)·x₀ + σ·ε, and the transformer's velocity prediction is scored against the rectified-flow target ε − x₀ (MSE over the tokens). It is the training objective measured on photographs the model never trains on; the same seed gives the same latents, noise and noise levels later, so the adapted number is a paired comparison, not a re-draw.

**CLIP-scored generations** (`pipe.generate` + `score_generations`): one image per training caption at fixed seeds (4 Euler steps, no guidance, 512²), scored by the frozen CLIP ViT-B/32 on prompt alignment (cosine × 100), zero-shot label accuracy (which of the six captions is nearest — an argmax with no threshold) and similarity to the mean embedding of the held-out real photographs of that species. `real_photo_baseline` scores the real test photographs the same way: the ceiling these numbers could reach. Look for: the load taking a few minutes, a flow-matching MSE around 0.5..1.5, label accuracy below the real photographs' 1.0 for at least some species, and a first grid of six generated birds.

In [ ]:
STEPS = 4  # @param {type:"integer"}
IMAGES_PER_PROMPT = 1  # @param {type:"integer"}
EVAL_SEED = 0

def grid(images, path, columns=6):
    tiles = [im.resize((256, 256)) for im in images]
    rows = (len(tiles) + columns - 1) // columns
    sheet = Image.new('RGB', (256 * columns, 256 * rows), 'white')
    for i, tile in enumerate(tiles):
        sheet.paste(tile, (256 * (i % columns), 256 * (i // columns)))
    sheet.save(path)
    return path

load_report = pipe.load_transformer()
print({**load_report, 'quantization': QUANTIZATION, 'parameters': count_parameters(pipe.transformer), 'lora_tensors': len(lora_parameter_names(pipe.transformer)), 'gpu_memory_gb': gpu_memory_gb()})
scorer = ClipScorer(weights_dir=SCORER_WEIGHTS_DIR, device=pipe.device)
t0 = time.perf_counter()
frozen_val = pipe.evaluate(val_records, seed=EVAL_SEED)
frozen_test = pipe.evaluate(test_records, seed=EVAL_SEED)
print({'frozen_flow_matching_mse': {'validation': frozen_val['flow_matching_mse'], 'test': frozen_test['flow_matching_mse']}, 'by_sigma_test': frozen_test['by_sigma'], 'seconds': round(time.perf_counter() - t0, 1)})

generation_prompts = [p for p in prompts for _ in range(IMAGES_PER_PROMPT)]
frozen_generation = pipe.generate(generation_prompts, seed=1000, steps=STEPS)
print({'generated': len(frozen_generation['images']), 'steps': frozen_generation['steps'], 'guidance_scale': frozen_generation['guidance_scale'], 'precision': frozen_generation['precision'], 'seconds': frozen_generation['seconds'], 'adapted': frozen_generation['model']['adapted']})
frozen_scores = score_generations(scorer, frozen_generation['images'], references=test_records)
real_ceiling = real_photo_baseline(scorer, test_records)
print({'frozen_generations': {k: frozen_scores[k] for k in ('clip_prompt_similarity', 'label_accuracy', 'reference_similarity')}})
print({'real_photo_ceiling': {k: real_ceiling[k] for k in ('clip_prompt_similarity', 'label_accuracy', 'reference_similarity')}})
for entry in frozen_scores['per_image'][::IMAGES_PER_PROMPT]:
    print({'prompt': entry['prompt'][:42], 'clip': entry['clip_prompt_similarity'], 'nearest': entry['nearest_prompt'][13:40], 'correct': entry['correct'], 'reference_similarity': entry['reference_similarity']})
print({'grid': str(grid([g['image'] for g in frozen_generation['images']], 'outputs/flux_schnell_generation_frozen_grid.jpg')), 'gpu_memory_gb': gpu_memory_gb()})

## 7. Bounded QLoRA fine-tuning

`pipe.adapt` trains the 380 LoRA tensors (rank 8, 9,338,880 parameters — 0.08 % of the transformer) that `peft` attached to the image-stream query, key, value and output projections of all 57 blocks, and nothing else; the 4-bit base, the VAE and the text encoders are frozen. Each step takes one training photograph's packed latent (VAE-encoded once, seeded), draws a noise level σ uniformly from (0, 1) and a noise tensor (both seeded), mixes x_σ = (1 − σ)·x₀ + σ·ε, and minimises the MSE between the predicted velocity and ε − x₀; AdamW at a fixed learning rate on float32 LoRA masters, gradient-norm clipping at 1.0, float16 autocast with loss scaling and gradient checkpointing (the 1,280-token activations of 57 blocks would not otherwise fit). Epoch 0 records the frozen model's validation loss, and the epoch with the lowest validation flow-matching loss is kept.

Watch the validation loss from epoch 0; three epochs over 36 images (108 steps) take on the order of half an hour on a T4, most of it the per-epoch validation pass. The training loss is a noisy per-step average over random noise levels and is not the quality signal — the paired held-out numbers in Section 8 are.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 1  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_flow_matching_mse': entry['val_loss']}
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=EVAL_SEED, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'steps': adapt_result['n_steps'], 'best_epoch': adapt_result['best_epoch'], 'precision': adapt_result['precision'], 'seconds': adapt_seconds, 'peak_gpu_memory_gb': round(torch.cuda.max_memory_allocated() / 1e9, 2)})

## 8. Held-out evaluation: the paired comparison

The test photographs were never used for training or epoch selection. The adapted model is scored exactly as the frozen model was in Section 6 — the same seed, so the same latents, noise and noise levels, and the same six prompt/seed pairs for generation — and the table puts the frozen, the adapted and the real-photo numbers side by side. The cell asserts only what the procedure guarantees — the kept epoch's validation loss is no higher than the frozen model's (epoch 0) and the re-scored validation loss matches the history — and prints the test comparison without a directional assertion: a lower test flow-matching MSE is what to look for, not what is promised. Look too for the generated birds moving towards the held-out photographs: higher reference similarity and label accuracy, and a second grid to compare with the first by eye. Six images per model from one seeded run give no dispersion estimate; these are sample-sanity numbers that show the adaptation contract works, not a benchmark, and CLIP agreement is not a human judgement of quality.

In [ ]:
adapted_val = pipe.evaluate(val_records, seed=EVAL_SEED)
adapted_test = pipe.evaluate(test_records, seed=EVAL_SEED)
adapted_generation = pipe.generate(generation_prompts, seed=1000, steps=STEPS)
adapted_scores = score_generations(scorer, adapted_generation['images'], references=test_records)
comparison = {
    'flow_matching_mse_validation': {'frozen': frozen_val['flow_matching_mse'], 'adapted': adapted_val['flow_matching_mse']},
    'flow_matching_mse_test': {'frozen': frozen_test['flow_matching_mse'], 'adapted': adapted_test['flow_matching_mse']},
    'flow_matching_mse_test_by_sigma': {s: {'frozen': frozen_test['by_sigma'][s], 'adapted': adapted_test['by_sigma'][s]} for s in adapted_test['by_sigma']},
    'clip_prompt_similarity': {'frozen': frozen_scores['clip_prompt_similarity'], 'adapted': adapted_scores['clip_prompt_similarity'], 'real_photos': real_ceiling['clip_prompt_similarity']},
    'label_accuracy': {'frozen': frozen_scores['label_accuracy'], 'adapted': adapted_scores['label_accuracy'], 'real_photos': real_ceiling['label_accuracy']},
    'reference_similarity': {'frozen': frozen_scores['reference_similarity'], 'adapted': adapted_scores['reference_similarity'], 'real_photos': real_ceiling['reference_similarity']},
}
for name, row in comparison.items():
    print({name: row})
for before, after in zip(frozen_scores['per_image'][::IMAGES_PER_PROMPT], adapted_scores['per_image'][::IMAGES_PER_PROMPT]):
    print({'prompt': before['prompt'][:42], 'reference_similarity': {'frozen': before['reference_similarity'], 'adapted': after['reference_similarity']}, 'correct': {'frozen': before['correct'], 'adapted': after['correct']}})
print({'grid': str(grid([g['image'] for g in adapted_generation['images']], 'outputs/flux_schnell_generation_adapted_grid.jpg'))})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY, 'staging': {'repo': STAGING_ID, 'revision': STAGING_REVISION}, 'quantization': QUANTIZATION, 'compute_dtype': pipe.compute_dtype},
    'scorer': frozen_scores['scorer'],
    'data_source': data_source,
    'dataset': dataset_report,
    'generation': {'steps': STEPS, 'guidance_scale': GUIDANCE_SCALE, 'images_per_prompt': IMAGES_PER_PROMPT, 'seed': 1000},
    'frozen': {'validation': frozen_val, 'test': frozen_test, 'generations': frozen_scores},
    'adapted': {'validation': adapted_val, 'test': adapted_test, 'generations': adapted_scores},
    'real_photo_ceiling': real_ceiling,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/flux_schnell_generation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
best = adapt_result['history'][adapt_result['best_epoch']]
assert best['val_loss'] <= adapt_result['history'][0]['val_loss']
assert abs(adapted_val['flow_matching_mse'] - best['val_loss']) < 1e-4
print({'test_flow_matching_mse_change': round(adapted_test['flow_matching_mse'] - frozen_test['flow_matching_mse'], 6), 'note': 'held-out observation, not asserted'})
print({'report': 'outputs/flux_schnell_generation_evaluation_report.json'})

## 9. A new prompt, artifact export and fresh reload

The adapted model renders `NEW_PROMPT` — a composition that appears in no training caption — at two seeds; the CLIP prompt similarity is printed as a sanity check, not an evaluation. One more image of the first training prompt is rendered and kept for the parity check.

`pipe.save_artifact` writes the 380 trained tensors (about 37 MB in float32) as `adapter.safetensors` with a `manifest.json` recording the artifact format, the base model's id, revision, licence, staging source and quantisation, the LoRA configuration, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). Two 4-bit transformers do not fit a 16 GB GPU, so the adapted pipeline then **releases** its transformer (`release_transformer`, which also forgets the adapter) before `FluxSchnellPipeline.from_artifact` re-verifies both snapshots, checks the manifest, the LoRA scope and the digest **before** deserialising, loads a fresh 4-bit transformer with the adapter attached and overlays the tensors — a new object from files, not the in-memory model (VER2). The fresh pipeline adopts the prompt embeddings already encoded (so the text encoders are not loaded again), and the cell asserts that it reproduces the held-out flow-matching loss recorded in Section 8 and the same image for the same prompt and seed (VER4: a mean absolute pixel difference below 1 on the 0..255 scale — same device, same kernels, same deterministic NF4 quantisation of the same bytes).

In [ ]:
import platform
import shutil

new_generation = pipe.generate([NEW_PROMPT, NEW_PROMPT], seed=2000, steps=STEPS)
new_scores = score_generations(scorer, new_generation['images'])
print({'new_prompt': NEW_PROMPT, 'clip_prompt_similarity': new_scores['clip_prompt_similarity'], 'seconds': new_generation['seconds'], 'note': 'sanity check, not an evaluation'})
for i, entry in enumerate(new_generation['images']):
    entry['image'].save(f'outputs/flux_schnell_generation_new_prompt_{i}.png')
before = pipe.generate([prompts[0]], seed=3000, steps=STEPS)['images'][0]['image']

artifact_dir = Path('outputs/flux_schnell_generation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'flux_schnell_generation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'quantization': artifact_manifest['base_model']['quantization']})

print({'adapted_transformer_released': pipe.release_transformer(), 'gpu_memory_gb': gpu_memory_gb()})
reloaded = FluxSchnellPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device, prompt_cache=cache)
reloaded_test = reloaded.evaluate(test_records, seed=EVAL_SEED)
after = reloaded.generate([prompts[0]], seed=3000, steps=STEPS)['images'][0]['image']
parity = {'flow_matching_mse_diff': round(abs(reloaded_test['flow_matching_mse'] - adapted_test['flow_matching_mse']), 8), 'mean_abs_pixel_diff': round(float(np.abs(np.asarray(before, dtype=np.float32) - np.asarray(after, dtype=np.float32)).mean()), 4)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch'], 'gpu_memory_gb': gpu_memory_gb()})
assert parity['flow_matching_mse_diff'] < 1e-5 and parity['mean_abs_pixel_diff'] < 1.0

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'precision': frozen_generation['precision'], 'source': pipe.source},
    'scorer': {**evaluation_report['scorer'], 'license': SCORER_LICENSE},
    'provenance': {
        'snapshots': {'model': len(MANIFEST['files']), 'scorer': len(SCORER_MANIFEST['files'])},
        'staging': MANIFEST['staging'],
        'safetensors_only': True,
        'remote_code_executed': False,
        'text_encoders_released_before_training': released,
        'transformer_loaded_after_encoding': load_report,
        'data_base_url': CORPUS_BASE_URL,
        'data_license': CORPUS_LICENSE,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'diffusers': diffusers.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'bitsandbytes': bitsandbytes.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    'data_source': data_source,
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/flux_schnell_generation_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

A LoRA of nine million parameters trained for half an hour on 36 photographs over a 4-bit 12 B base lowers the held-out flow-matching loss on twelve photographs the model never saw and moves its four-step generations towards the held-out real photographs of the same species. That is the claim: the adaptation contract teaches a rectified-flow transformer a narrow visual domain from a handful of captioned images on a 16 GB GPU, the held-out objective is measured on identical inputs before and after, and the artifact that carries the change is 37 MB.

The numbers are sample-sanity evidence. A flow-matching loss is the training objective, not a quality score; CLIP similarity and CLIP's nearest-caption vote are a frozen model's opinion, not a human judgement, and CLIP itself has biases about what a species name looks like; six images per model from one seeded run give no dispersion estimate; and nothing here measures aesthetics, diversity, artefacts or prompt fidelity beyond the six captions. Every number is the **4-bit** model's: NF4 quantisation changes the base's outputs relative to the bfloat16 checkpoint by an amount this notebook does not measure, and an adapter trained over the quantised base is meant to be served over it (the artifact manifest records `nf4`). Fine-tuning on a narrow domain can also erode the model elsewhere — the new prompt in Section 9 is a sanity check on one composition, not a test of generality.

Three things to carry to real data. **Captions are the contract:** the adapter learns the association between the caption text and the images; a caption that does not describe its image, or one caption for very different images, teaches noise. **Hold out by caption, not by image:** the split keeps every caption's images across sets so the held-out loss measures generalisation within the domain; a caption with one image cannot be evaluated. **Licences travel with the outputs:** FLUX.1 [schnell] is Apache-2.0 and the training photographs here are CC0 — with your own data, the rights to the images and to what the adapter produces are yours to establish.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can stage a pinned safetensors snapshot from a mirror and digest-verify it against a committed manifest, fetch and validate digest-pinned real photographs, encode prompts and release the encoders, load a 12 B transformer 4-bit, execute bounded QLoRA fine-tuning, evaluate the frozen and the adapted model on identical held-out inputs with a real-photo ceiling, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or image quality beyond the checks shown.

**Optional experiments (they do not affect the default path):** raise `EPOCHS` or `LEARNING_RATE` and watch the validation loss for the epoch where it turns; set `STEPS = 1` or `2` and read how few-step distillation trades prompt similarity for speed; set `IMAGES_PER_PROMPT = 2` for a less noisy label accuracy; or bring your own captioned photographs through BYOD and compare the real-photo ceiling with the adapted numbers.

## References

- Repository README: https://github.com/kurtvalcorza/flux-schnell-generation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/flux-schnell-generation-pipeline/blob/main/MODEL_CARD.md
- Weights notes (identity, mirror staging, byte-identity evidence): https://github.com/kurtvalcorza/flux-schnell-generation-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository (identity of record): https://huggingface.co/black-forest-labs/FLUX.1-schnell (revision `741f7c3ce8b383c54771c7003378a50191e9efe9`)
- Ungated mirror the bytes are staged from: https://huggingface.co/unsloth/FLUX.1-schnell (revision `9df3faa7ae3b6ddf0b2b69bb78616372897cc65c`)
- Black Forest Labs (2024). Announcing Black Forest Labs — FLUX.1: https://blackforestlabs.ai/announcing-black-forest-labs/ ; reference code: https://github.com/black-forest-labs/flux
- Liu, X., Gong, C., Liu, Q. (2023). Flow straight and fast: Learning to generate and transfer data with rectified flow. ICLR: https://arxiv.org/abs/2209.03003
- Sauer, A., et al. (2024). Fast high-resolution image synthesis with latent adversarial diffusion distillation: https://arxiv.org/abs/2403.12015
- Dettmers, T., et al. (2023). QLoRA: Efficient finetuning of quantized LLMs. NeurIPS: https://arxiv.org/abs/2305.14314
- Hu, E. J., et al. (2022). LoRA: Low-rank adaptation of large language models. ICLR: https://arxiv.org/abs/2106.09685
- Cherti, M., et al. (2023). Reproducible scaling laws for contrastive language-image learning. CVPR (the LAION CLIP scorer): https://arxiv.org/abs/2212.07143
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)